<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.es/cap03/cap03_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 3 Operaciones Espaciales: Intensidad, Histograma y Filtrado

Este capítulo profundiza en el procesamiento de imágenes en el dominio espacial, partiendo de la manipulación directa de píxeles e histogramas para el realce de contraste, hasta la aplicación de filtros locales por convolución para suavizado, reducción de ruido y detección de bordes. El objetivo es desarrollar la intuición matemática y computacional que sustenta gran parte de los algoritmos modernos de Visión Computacional.

## 3.1 Objetivos

Al final de este capítulo, usted será capaz de:

* **Manipular intensidad y píxeles:** Ejecutar operaciones aritméticas saturadas (`mm::addm`, `mm::subm`) y lógicas bit a bit (`mm::band`, `mm::bor`, `mm::bnot`) para combinación y selección de regiones de interés (ROI), y aplicar *alpha blending* (`mm::blend`) para fusión ponderada de imágenes;
* **Procesar histogramas:** Interpretar el histograma como diagnóstico tonal y aplicar ecualización global mediante CDF (`mm::equalize`); en la ruta Python, también la ecualización adaptativa (CLAHE) y la especificación de histograma para transferencia de perfil tonal entre imágenes;
* **Comprender fundamentos espaciales:** Entender vecindad, *padding* de borde (`mm::pad`) y la diferencia entre correlación cruzada (`mm::conv`) y convolución — incluyendo por qué *kernels* asimétricos como el de Sobel producen resultados distintos en las dos operaciones;
* **Aplicar filtrado de suavizado:** Usar el filtro de media (`mm::blur`, o `mm::conv` con *kernel* uniforme) y el filtro Gaussiano (`mm::gaussian`) para reducción de ruido, comprendiendo la ventaja de la ponderación radial y de la separabilidad Gaussiana;
* **Aplicar filtrado de realce:** Usar el Laplaciano $w_4$ y $w_8$ (`mm::laplacian`) para realce isotrópico de bordes, el operador de Sobel (`mm::sobel`) para la magnitud del gradiente — y, en la ruta Python, la descomposición direccional $G_x$, $G_y$ y ángulo —, y el *Unsharp Masking* (`mm::usm`) para amplificación de alta frecuencia controlada por el parámetro $k$;
* **Utilizar filtros de orden:** Aplicar el filtro de la mediana (`mm::median`) para eliminación de ruido sal y pimienta, comprendiendo por qué su naturaleza no lineal y la robustez a *outliers* lo hacen superior a los filtros lineales en ese escenario;
* **Resolver problemas prácticos:** Encadenar técnicas en *pipelines* de preprocesamiento (ecualización → Gaussiano → Canny; con CLAHE en lugar de la ecualización en la ruta Python) y usar las funciones de `morph` (`mm::conv`, `mm::histImg`, `mm::equalize`, `mm::drawImgKernel`) para análisis y visualización didáctica de cada etapa.

## 3.2 Operaciones a Nivel de Intensidad

El nivel más elemental de procesamiento de imágenes actúa directamente sobre los valores de los píxeles, sin considerar la vecindad. Estas operaciones —llamadas **transformaciones de punto** (*point operations*)— son las más rápidas computacionalmente y forman la base para técnicas más complejas.

Formalmente, una transformación de punto puede describirse como:

<a id="eq-03-ponto"></a>
$$
g(x,y) = T[f(x,y)] \tag{3.1}
$$


donde $f(x,y)$ es la imagen de entrada, $g(x,y)$ es la salida y $T$ es una función aplicada a cada píxel individualmente.

### 3.2.1 Preparando el Entorno Práctico

El siguiente bloque carga la biblioteca `morph` del repositorio (el módulo `morph.py` y, en la ruta C++, también el `morph.hpp` utilizado en el `#include` de las celdas compiladas).

In [ ]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build de la pista C++ (.cpp, binario, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# El kernel es Python incluso en la pista C++: `mm` (morph.py) es usado por los
# simuladores, por la visualización de las figuras que el binario C++ genera y por el
# estado mm::Image entre celdas. cpp=True también descarga la pista compilada
# (morph.hpp + stb_image*.h), usada en el #include de las celdas %%writefile *.cpp.
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

Como objeto de estudio a lo largo de este capítulo, utilizaremos las imágenes de vida silvestre presentadas en las [Figura 3.1](#fig-03-mandrill) y [Figura 3.3](#fig-03-leopardo).. A partir de ellas, exploraremos operaciones espaciales sobre intensidad, histogramas y filtrado, analizando sus efectos en el realce, la suavización, la reducción de ruido y la detección de bordes, con el fin de comprender los fundamentos matemáticos y computacionales del PDI.

In [ ]:
%%writefile tmp/fig_03_mandrill.cpp
#define MM_OUT "tmp/fig_03_mandrill.png"
//| label: fig-03-mandrill
//| fig-cap: "*Mandrill* (*Mandrillus sphinx*) fotografado em ambiente natural na África do Sul. Crédito: Carlos Guilherme Rodrigues (CC BY-SA 3.0)."
//| echo: true

#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    std::string base    = "https://upload.wikimedia.org/wikipedia/commons";
    std::string arquivo = "Carlos_Guilherme_Rodrigues_%2876515283%29.jpeg";
    std::string url     = base + "/9/9b/" + arquivo;

    mm::Image img_color = mm::read(url);
    mm::Image img_gray  = mm::gray(img_color);

    mm::show(img_color, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_8.png");
// [pdi:state-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_mandrill.cpp -o tmp/fig_03_mandrill \
  && ./tmp/fig_03_mandrill \
  && test -f "tmp/fig_03_mandrill.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_mandrill.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_mandrill.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_mandrill.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.1:** *Mandrill* (*Mandrillus sphinx*) fotografado em ambiente natural na África do Sul. Crédito: Carlos Guilherme Rodrigues (CC BY-SA 3.0).


### 3.2.2 Operaciones Aritméticas

Las operaciones aritméticas entre imágenes se utilizan ampliamente en PDI para combinar, comparar o realzar información. La **resta de imágenes** es especialmente poderosa para detectar diferencias entre dos cuadros — por ejemplo, en la eliminación del fondo estático en cámaras de vigilancia:

<a id="eq-03-subtracao"></a>
$$
g(x,y) = f_1(x,y) - f_2(x,y) \tag{3.2}
$$


La **suma saturada** limita el resultado al intervalo $[0, 255]$: los valores superiores a 255 se fijan en 255, evitando el *overflow* silencioso del tipo `uint8` (p. ej., $200 + 100 = 44$ en lugar de 300). La **resta saturada** aplica el mismo principio por el lado inferior: los valores negativos se fijan en 0.

> ### ⚠️ Saturación y *overflow*
>
> Las operaciones aritméticas en `uint8` sufren *overflow* silencioso: $200 + 100 = 44$ (no 300). `mm::addm` y `mm::subm` realizan la **saturación automática**, fijando el resultado en $[0, 255]$. El *blending* utiliza pesos fraccionarios: `mm::blend` opera internamente en punto flotante y solo entonces redondea y satura a `uint8`.

La [Figura 3.2](#fig-03-aritmetica) demuestra la suma de una constante (aclarado) y la resta de una constante (oscurecimiento con saturación en 0).

In [ ]:
%%writefile tmp/fig_03_aritmetica.cpp
#define MM_OUT "tmp/fig_03_aritmetica.png"
//| label: fig-03-aritmetica
//| fig-cap: "Operações aritméticas saturadas: adição de constante (clareamento) e subtração de constante (escurecimento com saturação em 0)."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    int fundo = 60;

    mm::Image img_add = mm::addm(img_gray, fundo);
    mm::Image img_sub = mm::subm(img_gray, fundo);

    mm::show(
        std::vector<mm::Image>{img_gray, img_add, img_sub},
        MM_OUT,
        std::vector<std::string>{"Original", "addm (+60)", "subm (−60)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_aritmetica_0.png");
mm::write(img_add, "tmp/fig_03_aritmetica_1.png");
mm::write(img_sub, "tmp/fig_03_aritmetica_2.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_aritmetica.cpp -o tmp/fig_03_aritmetica \
  && ./tmp/fig_03_aritmetica \
  && test -f "tmp/fig_03_aritmetica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_aritmetica.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_aritmetica_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_aritmetica_0.png"),
            mm.read("tmp/fig_03_aritmetica_1.png"),
            mm.read("tmp/fig_03_aritmetica_2.png"),
        ],
        titles=[
            'Original',
            'addm (+60)',
            'subm (−60)',
        ],
        cols=3,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.2:** Operações aritméticas saturadas: adição de constante (clareamento) e subtração de constante (escurecimento com saturação em 0).


### 3.2.3 Mezcla Ponderada (*Alpha Blending*)

La **mezcla ponderada** (*alpha blending*) combina dos imágenes utilizando pesos complementarios $\alpha$ y $(1-\alpha)$:

<a id="eq-03-blend"></a>
$$
g(x,y) = \alpha\,f_1(x,y) + (1-\alpha)\,f_2(x,y), \quad \alpha \in [0,1] \tag{3.3}
$$


Cuando $\alpha = 1$, se obtiene únicamente la imagen $f_1$; cuando $\alpha = 0$, solo $f_2$. Los valores intermedios producen una transición suave entre ambas, siendo ampliamente utilizados en composición de imágenes, superposición de capas, marcas de agua y efectos de fusión visual.

Para que la combinación produzca un resultado coherente, es necesario alinear previamente las regiones de interés. En la [Figura 3.4](#fig-03-blend), se recorta el rostro del leopardo con `mm::crop(img_leop_gray, 250, H-300, 100, W-200)` y la región facial del mandril con `mm::crop(img_gray, 100, 400, 380, 530)`, de modo que los ojos y la estructura facial queden aproximadamente alineados. El recorte del leopardo se redimensiona entonces (`mm::resize`) a las dimensiones del mandril antes de la mezcla.

`mm::blend` realiza la operación en punto flotante — evitando *overflow* en los cálculos con pesos fraccionarios — y solo entonces redondea y satura el resultado a `uint8`.

In [ ]:
%%writefile tmp/fig_03_leopardo.cpp
#define MM_OUT "tmp/fig_03_leopardo.png"
// Compile: g++ -std=c++17 -o programa programa.cpp -I. -lcurl -lpng -ljpeg

#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    //| label: fig-03-leopardo
    //| fig-cap: "Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0)."
    //| echo: true

    mm::Image img_leop = mm::read("https://upload.wikimedia.org/wikipedia/commons/9/92/Leopard_%28Panthera_pardus%29_portrait.jpg");
    mm::Image img_leop_gray = mm::gray(img_leop);

    mm::show(img_leop, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_leop, "tmp/state/img_leop_12.png");
mm::write(img_leop_gray, "tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_leopardo.cpp -o tmp/fig_03_leopardo \
  && ./tmp/fig_03_leopardo \
  && test -f "tmp/fig_03_leopardo.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_leopardo.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_leopardo.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_leopardo.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.3:** Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0).


In [ ]:
%%writefile tmp/fig_03_blend.cpp
#define MM_OUT "tmp/fig_03_blend.png"
//| label: fig-03-blend
//| fig-cap: "*Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    // recortes alinhados: rosto do leopardo e região facial do mandril
    mm::Image leo = mm::crop(img_leop_gray, 250, img_leop_gray.h - 300, 100, img_leop_gray.w - 200);
    mm::Image mandrill = mm::crop(img_gray, 100, 400, 380, 530);
    mm::Image leo_r = mm::resize(leo, mandrill.w, mandrill.h, "bilinear");

    mm::show(
        std::vector<mm::Image>{mm::blend(mandrill, leo_r, 1.0), mm::blend(mandrill, leo_r, 0.8),
                               mm::blend(mandrill, leo_r, 0.6), mm::blend(mandrill, leo_r, 0.4),
                               mm::blend(mandrill, leo_r, 0.2), mm::blend(mandrill, leo_r, 0.0)},
        MM_OUT,
        std::vector<std::string>{"α=1.0", "α=0.8", "α=0.6", "α=0.4", "α=0.2", "α=0.0"},
        6
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_blend.cpp -o tmp/fig_03_blend \
  && ./tmp/fig_03_blend \
  && test -f "tmp/fig_03_blend.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_blend.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_blend.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_blend.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.4:** *Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente.


### 3.2.4 Operaciones Lógicas y Máscaras Bit a Bit

Las operaciones lógicas bit a bit (AND, OR y NOT) actúan directamente sobre los bits de cada píxel y son la base para la creación y aplicación de **máscaras** (*masks*) —imágenes binarias con solo 0 (negro) y 255 (blanco) utilizadas para aislar **Regiones de Interés** (**ROI**).

El comportamiento de cada operación se deriva de la representación binaria del 255 (`11111111`) y del 0 (`00000000`):

- **AND** con la máscara: donde $m = 255$, los bits originales se preservan; donde $m = 0$, el píxel se pone a cero. Resultado: recorte de la ROI.
<a id="eq-03-mascara"></a>
$$
g(x,y) = f(x,y) \;\text{AND}\; m(x,y) \tag{3.4}
$$

- **OR** con la máscara: donde $m = 255$, el píxel se fuerza a blanco; donde $m = 0$, se mantiene el valor original. Resultado: iluminación de la ROI.
- **NOT** (sin máscara): invierte todos los bits ($g = 255 - f$), produciendo el negativo fotográfico de la imagen.

La [Figura 3.5](#fig-03-logica) ilustra las tres operaciones aplicadas a la imagen del mandril con una máscara circular.

In [ ]:
%%writefile tmp/fig_03_logica.cpp
#define MM_OUT "tmp/fig_03_logica.png"
//| label: fig-03-logica
//| fig-cap: "Operações lógicas bit a bit com máscara circular: AND (isolamento da ROI), OR (iluminação da ROI) e NOT (negativo)."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    // img_gray ya está inicializado (proporcionado automáticamente)

    int h = img_gray.h;
    int w = img_gray.w;

    // Máscara circular preenchida, centrada na imagem (mm.circle desenha o
    // disco; a versão didática, teste de raio pixel a pixel, é mm.circle0).
    mm::Image mask_circ(h, w);
    int radius = std::min(h, w) / 3 - 10;
    mask_circ = mm::circle(mask_circ, w / 2, h / 2, radius, 255, -1);

    // Operações via morph
    mm::Image img_not = mm::bnot(img_gray);            // NOT: negativo fotográfico
    mm::Image img_and = mm::band(img_gray, mask_circ); // preserva apenas a ROI circular
    mm::Image img_or  = mm::bor(img_gray, mask_circ);  // ilumina a região da máscara

    mm::show(
        std::vector<mm::Image>{img_gray, img_and, img_or, img_not},
        MM_OUT,
        std::vector<std::string>{"Original", "AND (ROI circular)", "OR (ilumina ROI)", "NOT (negativo)"},
        4
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_logica_0.png");
mm::write(img_and, "tmp/fig_03_logica_1.png");
mm::write(img_or, "tmp/fig_03_logica_2.png");
mm::write(img_not, "tmp/fig_03_logica_3.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_logica.cpp -o tmp/fig_03_logica \
  && ./tmp/fig_03_logica \
  && test -f "tmp/fig_03_logica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_logica.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_logica_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_logica_0.png"),
            mm.read("tmp/fig_03_logica_1.png"),
            mm.read("tmp/fig_03_logica_2.png"),
            mm.read("tmp/fig_03_logica_3.png"),
        ],
        titles=[
            'Original',
            'AND (ROI circular)',
            'OR (ilumina ROI)',
            'NOT (negativo)',
        ],
        cols=4,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.5:** Operações lógicas bit a bit com máscara circular: AND (isolamento da ROI), OR (iluminação da ROI) e NOT (negativo).


## 3.3 Histograma de Imágenes

El **histograma** de una imagen en tonos de gris es una función discreta que describe la distribución de frecuencias de las intensidades:

<a id="eq-03-histograma"></a>
$$
h(r_k) = n_k, \quad k = 0, 1, \ldots, L-1 \tag{3.5}
$$


donde $r_k$ es el $k$-ésimo nivel de intensidad, $n_k$ es el número de píxeles con esa intensidad y $L$ es el total de niveles (típicamente 256 para 8 bits). El histograma normalizado estima la probabilidad de cada nivel:

<a id="eq-03-hist-norm"></a>
$$
p(r_k) = \frac{n_k}{MN} \tag{3.6}
$$


donde $MN$ es el total de píxeles. Por ser una **estadística global**, el histograma no contiene información posicional, pero revela características esenciales como brillo medio, contraste y distribución tonal. En la práctica, `mm::hist(img)` devuelve el vector de conteos $h(r_k)$, que sirve tanto para visualización (mediante `mm::histImg`) como para cálculos como **función de distribución acumulada (CDF)** y ecualización.

> ### 📝 Interpretación del Histograma
>
> - **Estrecho a la izquierda:** imagen subexpuesta (oscura).
> - **Estrecho a la derecha:** imagen sobreexpuesta (clara).
> - **Concentrado en el centro:** bajo contraste.
> - **Distribuido por todo el rango:** alto contraste, buena utilización de los tonos disponibles.

La [Figura 3.6](#fig-03-histograma) presenta el histograma de la imagen del mandril, así como versiones oscurecida (`mm::subm`) y aclarada (`mm::addm`). Se observa el desplazamiento de la distribución de intensidades hacia la izquierda y hacia la derecha, respectivamente. Nótese que el intervalo representado en el eje $x$ no corresponde necesariamente a todo el rango de 0 a 255.

In [ ]:
%%writefile tmp/fig_03_histograma.cpp
#define MM_OUT "tmp/fig_03_histograma.png"
//| label: fig-03-histograma
//| fig-cap: "Histogramas da imagem original, de uma versão escurecida (−80) e de uma clareada (+80). A subtração/adição satura em 0 e 255."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    // img_gray is provided already initialized

    mm::Image img_dark = mm::subm(img_gray, 80);
    mm::Image img_high = mm::addm(img_gray, 80);

    mm::show(
        std::vector<mm::Image>{img_gray, img_dark, img_high,
             mm::histImg(img_gray), mm::histImg(img_dark), mm::histImg(img_high)},
        MM_OUT,
        std::vector<std::string>{"Original", "Escurecida (-80)", "Clareada (+80)",
            "Histograma - original", "Histograma - escurecida", "Histograma - clareada"},
        3
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_histograma.cpp -o tmp/fig_03_histograma \
  && ./tmp/fig_03_histograma \
  && test -f "tmp/fig_03_histograma.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_histograma.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_histograma.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_histograma.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.6:** Histogramas da imagem original, de uma versão escurecida (−80) e de uma clareada (+80). A subtração/adição satura em 0 e 255.


### 3.3.1 Ecualización de Histograma

La **ecualización de histograma** redistribuye las intensidades para que el histograma resultante sea lo más uniforme posible. El mapeo está dado por la **función de distribución acumulada (CDF)**:

<a id="eq-03-equalizacao"></a>
$$
s_k = T(r_k) = (L-1)\sum_{j=0}^{k} p(r_j) = \frac{L-1}{MN}\sum_{j=0}^{k} n_j \tag{3.7}
$$


La transformación es monótona: los niveles frecuentes reciben intervalos mayores en el dominio de salida (mayor separación → más contraste), mientras que los niveles raros se comprimen.

El algoritmo completo, en cinco etapas, se presenta en la [Tabela 3.1](#tbl-03-equalizacao).

<a id="tbl-03-equalizacao"></a>

**Tabela 3.1:** Algoritmo de ecualización de histograma.

| Etapa | Operación | Fórmula |
|:-----:|:---------|:--------|
| 1 | **Histograma** | $h[k] \leftarrow$ número de píxeles con intensidad $k$, $k=0\ldots L-1$ |
| 2 | **Probabilidad** | $p[k] \leftarrow h[k] / MN$ |
| 3 | **CDF** | $\text{cdf}[k] \leftarrow \sum_{j=0}^{k} p[j]$ (suma acumulada) |
| 4 | ***Look-Up Table* (mapeo)** | $\text{lut}[k] \leftarrow \text{round}(\text{cdf}[k] \times (L-1))$ |
| 5 | **Aplicación** | $g[i,j] \leftarrow \text{lut}[f[i,j]]$ (para todo píxel) |


Observe en la [Figura 3.7](#fig-03-equalizacao-didatica) que la ecualización **redistribuye** los tonos existentes a posiciones más espaciadas en el rango $[0, L-1]$, pero no crea tonos nuevos — la imagen ecualizada continúa con exactamente 3 tonos distintos, ahora en $\{1, 5, 7\}$ en lugar de $\{2, 3, 4\}$.

In [ ]:
%%writefile tmp/fig_03_equalizacao_didatica.cpp
#define MM_OUT "tmp/fig_03_equalizacao_didatica.png"
//| label: fig-03-equalizacao-didatica
//| fig-cap: "Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. *mm::equalize(img, 3)* faz o mapeamento."
//| echo: true
//| output: true
#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
    mm::Image img5(5, 5);
    int vals[5][5] = {{3, 4, 2, 3, 4},
                      {4, 3, 3, 4, 3},
                      {2, 3, 4, 3, 2},
                      {3, 4, 3, 2, 3},
                      {4, 3, 2, 3, 4}};
    for (int y = 0; y < 5; y++) {
        for (int x = 0; x < 5; x++) {
            img5.at(y, x) = vals[y][x];
        }
    }

    mm::Image img5_eq = mm::equalize(img5, 3);   // L = 2^3 = 8

    std::cout << "Imagem original 5x5 (3 bits):\n";
    std::cout << mm::drawImg(img5);
    std::cout << "Imagem equalizada 5x5:\n";
    std::cout << mm::drawImg(img5_eq);

    mm::show(std::vector<mm::Image>{img5, img5_eq, mm::histImg(img5), mm::histImg(img5_eq)},
             MM_OUT,
             std::vector<std::string>{"Original", "Equalizada", "Histograma - original", "Histograma - equalizada"},
             2);
    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_equalizacao_didatica.cpp -o tmp/fig_03_equalizacao_didatica \
  && ./tmp/fig_03_equalizacao_didatica \
  && test -f "tmp/fig_03_equalizacao_didatica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_equalizacao_didatica.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_equalizacao_didatica.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_equalizacao_didatica.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.7:** Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. *mm::equalize(img, 3)* faz o mapeamento.


**Limitación:** la ecualización global puede realzar en exceso los ruidos y producir contraste excesivo en regiones homogéneas. El **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) reduce este problema al aplicar la ecualización en bloques locales (*tiles*) y limitar la altura de los picos del histograma antes de la ecualización.

La [Figura 3.8](#fig-03-equalizacao) compara la imagen original, la ecualización global mediante `mm::equalize` y el CLAHE de OpenCV, mostrando también los histogramas resultantes. A diferencia de la ecualización global, que utiliza una única transformación basada en la CDF de toda la imagen, el CLAHE adapta el contraste a cada región, siendo particularmente útil en imágenes con iluminación no uniforme.

En el ejemplo, se utilizó `clipLimit=2.0` y `tileGridSize=(32,32)`. El parámetro `clipLimit` define cuánto pueden crecer los picos del histograma local antes de ser recortados (*clipped*). En OpenCV, este valor es un factor relativo: el límite real se calcula aproximadamente como `clipLimit × (número de píxeles del bloque / número de niveles de gris)`. Por ejemplo, en un bloque con 4096 píxeles y una imagen de 8 bits (256 niveles de gris), la frecuencia media por nivel es $4096/256=16$. Así, `clipLimit=2.0` permite picos de aproximadamente $2\times16=32$ ocurrencias antes del recorte. Las ocurrencias excedentes no se descartan: se redistribuyen entre los demás niveles de gris del histograma, reduciendo la concentración excesiva en pocos niveles y evitando una amplificación exagerada del contraste local. Valores menores limitan más el contraste y reducen la amplificación del ruido, mientras que valores mayores permiten un realce más intenso, pero pueden introducir artefactos.

- `clipLimit=1.0`: realce suave y conservador;
- `clipLimit=2.0`: buen equilibrio entre contraste y naturalidad;
- `clipLimit=4.0`: mayor énfasis en los detalles locales;
- `clipLimit=8.0`: contraste agresivo, con posible amplificación del ruido.

Así, el CLAHE suele producir resultados más naturales que la ecualización global, especialmente en imágenes con sombras, reflejos o iluminación desigual.

In [ ]:
%%writefile tmp/fig_03_equalizacao.cpp
#define MM_OUT "tmp/fig_03_equalizacao.png"
//| label: fig-03-equalizacao
//| fig-cap: "Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    mm::Image img_eq = mm::equalize(img_gray);

    mm::show(
        std::vector<mm::Image>{img_gray, img_eq, mm::histImg(img_gray), mm::histImg(img_eq)},
        MM_OUT,
        std::vector<std::string>{"Original", "mm.equalize (CDF)", "Histograma - original", "Histograma - equalizado"},
        2
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_equalizacao.cpp -o tmp/fig_03_equalizacao \
  && ./tmp/fig_03_equalizacao \
  && test -f "tmp/fig_03_equalizacao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_equalizacao.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_equalizacao.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_equalizacao.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.8:** Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp.


### 3.3.2 Especificación de Histograma

Mientras que la ecualización impone una distribución uniforme, la **especificación de histograma** (*histogram matching*) permite que el histograma de la imagen de salida siga una distribución **arbitraria** — por ejemplo, el histograma de otra imagen de referencia.

El procedimiento involucra tres etapas:

1. Calcular la CDF de la imagen de entrada: $P_r(r_k)$.
2. Calcular la CDF de la imagen de referencia: $P_z(z_k)$.
3. Para cada nivel $r_k$, encontrar el nivel $z$ que minimiza $|P_z(z) - P_r(r_k)|$.

<a id="eq-03-especificacao"></a>
$$
T(r_k) = \arg\min_{z}\,|P_z(z) - P_r(r_k)| \tag{3.8}
$$


En la [Figura 3.9](#fig-03-especificacao), transferimos el perfil tonal del leopardo ([Figura 3.3](#fig-03-leopardo)) a la imagen del mandril — una aplicación directa del concepto visto en el *blending*: en lugar de fusionar píxeles, aquí fusionamos distribuciones tonales.

In [ ]:
%%writefile tmp/fig_03_especificacao.cpp
#define MM_OUT "tmp/fig_03_especificacao.png"
//| label: fig-03-especificacao
//| fig-cap: "Especificação de histograma (mapear o mandril para o perfil tonal do leopardo) precisa da CDF inversa da referência — fica só na trilha Python. Aqui, a equalização global do mandril, como comparação."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    mm::Image img_eq = mm::equalize(img_gray);

    mm::show(std::vector<mm::Image>{img_gray, img_leop_gray, img_eq},
             MM_OUT,
             std::vector<std::string>{"Mandril (original)", "Leopardo (referencia)", "Mandril equalizado"},
             3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_especificacao_0.png");
mm::write(img_leop_gray, "tmp/fig_03_especificacao_1.png");
mm::write(img_eq, "tmp/fig_03_especificacao_2.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_especificacao.cpp -o tmp/fig_03_especificacao \
  && ./tmp/fig_03_especificacao \
  && test -f "tmp/fig_03_especificacao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_especificacao.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_especificacao_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_especificacao_0.png"),
            mm.read("tmp/fig_03_especificacao_1.png"),
            mm.read("tmp/fig_03_especificacao_2.png"),
        ],
        titles=[
            'Mandril (original)',
            'Leopardo (referencia)',
            'Mandril equalizado',
        ],
        cols=3,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.9:** Especificação de histograma (mapear o mandril para o perfil tonal do leopardo) precisa da CDF inversa da referência — fica só na trilha Python. Aqui, a equalização global do mandril, como comparação.


## 3.4 Fundamentos Espaciales: Vecindad, Convolución y *Kernels*

Las operaciones de filtrado espacial no actúan sobre un píxel aislado, sino sobre una **vecindad** que lo rodea. Para ello, se utiliza una pequeña matriz de coeficientes denominada **kernel** (o máscara), que recorre toda la imagen mediante una **ventana deslizante** (*sliding window*).

Las ventanas más comunes son de 3×3, 5×5 y 7×7. En una ventana de 3×3, por ejemplo, el píxel central se procesa junto con sus ocho vecinos inmediatos. En cada posición de la ventana, los valores de los píxeles se combinan con los coeficientes del *kernel*, produciendo un nuevo valor para el píxel central.

### 3.4.1 Vecindad

Considere una ventana de 3×3 centrada en el píxel $(x,y)$:

<a id="eq-03-box3x3"></a>
$$
\begin{bmatrix}
(x-1,y-1) & (x,y-1) & (x+1,y-1) \\
(x-1,y)   & (x,y)   & (x+1,y)   \\
(x-1,y+1) & (x,y+1) & (x+1,y+1)
\end{bmatrix} \tag{3.9}
$$


De forma general, una ventana de tamaño $(2a+1)\times(2b+1)$ abarca todos los píxeles situados hasta $a$ posiciones en horizontal y hasta $b$ posiciones en vertical con respecto al píxel central. Así, una ventana de 3×3 corresponde a $a=b=1$, una ventana de 5×5 a $a=b=2$, y así sucesivamente.

Matemáticamente, la vecindad se define como

<a id="eq-03-vizinhanca"></a>
$$
\mathcal{V}(x,y)=
\{(x+s,\,y+t): -a\le s\le a,\,-b\le t\le b\} \tag{3.10}
$$


### 3.4.2 Tratamiento de Bordes

Los píxeles cercanos a los bordes tienen parte de su vecindad fuera de la imagen. Para aplicar filtros en estas regiones, es necesario definir cómo se obtendrán los valores externos. Las tres estrategias más comunes (con la constante equivalente de OpenCV entre paréntesis) son:

- **Zero-padding** (`BORDER_CONSTANT`): completa la región externa con ceros.
- **Replicación** (`BORDER_REPLICATE`): repite el valor del píxel del borde.
- **Reflexión** (`BORDER_REFLECT_101`): refleja los píxeles vecinos, sin repetir el del borde.

`mm::conv` usa la reflexión por defecto, ya que preserva mejor la continuidad de los niveles de gris y reduce artefactos en el tratamiento de los bordes.

El siguiente ejemplo compara las tres estrategias con `mm::pad` en una matriz 3×3. Observa cómo cada una rellena los píxeles externos necesarios para aplicar un filtro 3×3 también en las esquinas.

In [ ]:
%%writefile tmp/mm_out_1.cpp
// Compile with: g++ -std=c++17 -o program program.cpp -I. -lm

#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
    // Construir la imagen 3x3
    mm::Image img(3, 3);
    int valores[3][3] = {{1,2,3},{4,5,6},{7,8,9}};
    for (int y = 0; y < 3; y++)
        for (int x = 0; x < 3; x++)
            img.at(y, x) = valores[y][x];

    std::vector<std::string> bordas = {"constant", "replicate", "reflect101"};

    std::cout << "Imagem original:\n";
    std::cout << mm::drawImg(img) << "\n";

    for (int k : {3, 5}) {
        int b = k / 2;
        std::cout << "=== Kernel " << k << "x" << k << " (padding b=" << b << ") ===\n";
        for (const auto& nome : bordas) {
            std::cout << nome << "\n";
            mm::Border border;
            if (nome == "constant")
                border = mm::Border::CONSTANT;
            else if (nome == "replicate")
                border = mm::Border::REPLICATE;
            else
                border = mm::Border::REFLECT101;
            std::cout << mm::drawImg(mm::pad(img, b, border)) << "\n";
        }
    }

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/mm_out_1.cpp -o tmp/mm_out_1 \
  && ./tmp/mm_out_1

Note que el resultado de un filtro puede variar significativamente según el tratamiento adoptado para los bordes de la imagen.

En `morph.hpp`, las funciones de filtrado (`mm::conv`, `mm::blur`, `mm::gaussian`, `mm::laplacian`, `mm::usm`) aplican *padding* por **reflexión** (`mm::Border::REFLECT101`) por defecto — el mismo comportamiento que `cv2.filter2D`. La variante didáctica `mm::conv0` usa `mm::Border::KEEP`: los píxeles del borde mantienen el valor original, sin el filtro. En cambio, `mm::sobel` y `mm::prewitt` dejan el borde en cero (calculan solo el interior).

### 3.4.3 Correlación vs. Convolución

Existen dos mecanismos matemáticamente relacionados.

**Correlación cruzada** (*cross-correlation*) — el *kernel* se aplica directamente:

<a id="eq-03-correlacao"></a>
$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x+s,\,y+t) \tag{3.11}
$$


**Convolución bidimensional** — el *kernel* se rota 180° antes de su aplicación:

<a id="eq-03-convolucao"></a>
$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x-s,\,y-t) \tag{3.12}
$$


Para *kernels* simétricos (Gaussiano, Laplaciano, media) las dos operaciones producen resultados idénticos. Para *kernels* asimétricos (Sobel, Prewitt) la diferencia es significativa, como lo muestran los ejemplos a continuación.

#### 3.4.3.1 Correlación (`mm::conv`)

In [ ]:
%%writefile tmp/mm_out_2.cpp
// Converte código Python para C++ (usando morph.hpp)
// Compile com: g++ -std=c++17 programa.cpp -o programa $(pkg-config --cflags --libs opencv4)

#include "morph.hpp"
#include <iostream>

int main() {
    // imagem 4x4 (uint8) e kernel assimétrico 3x3
    mm::Image img(4, 4);
    // Preenche com os valores da matriz
    {
        int valores[4][4] = {{1, 2, 3, 4}, {5, 6, 7, 8}, {9, 10, 11, 12}, {13, 14, 15, 16}};
        for (int y = 0; y < 4; y++) {
            for (int x = 0; x < 4; x++) {
                img.at(y, x) = static_cast<unsigned char>(valores[y][x]);
            }
        }
    }

    mm::Kernel w{{0,1,2},{0,0,0},{0,0,0}};

    mm::Image corr = mm::conv(img, w, mm::Border::CONSTANT);  // zero fora da imagem

    std::cout << "Imagem original:\n";         std::cout << mm::drawImg(img);
    std::cout << "Kernel:\n";                  std::cout << mm::drawImg(w);
    std::cout << "Resultado da correlação:\n"; std::cout << mm::drawImg(corr);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/mm_out_2.cpp -o tmp/mm_out_2 \
  && ./tmp/mm_out_2

`mm::conv` realiza correlación, es decir, aplica el *kernel* exactamente en la orientación proporcionada.

#### 3.4.3.2 Convolución

In [ ]:
%%writefile tmp/mm_out_3.cpp
// Compile: g++ -std=c++17 -o program program.cpp -I. && ./program
#include "morph.hpp"
#include <iostream>

int main() {
    // repetidos aqui para a célula ser independente
    mm::Image img(4, 4);
    img.at(0,0) = 1;   img.at(0,1) = 2;   img.at(0,2) = 3;   img.at(0,3) = 4;
    img.at(1,0) = 5;   img.at(1,1) = 6;   img.at(1,2) = 7;   img.at(1,3) = 8;
    img.at(2,0) = 9;   img.at(2,1) = 10;  img.at(2,2) = 11;  img.at(2,3) = 12;
    img.at(3,0) = 13;  img.at(3,1) = 14;  img.at(3,2) = 15;  img.at(3,3) = 16;

    mm::Kernel w{{0,1,2},{0,0,0},{0,0,0}};

    // kernel rotacionado 180° (equivale a np.rot90(w, 2))
    mm::Kernel w_conv{{0,0,0},{0,0,0},{2,1,0}};

    mm::Image conv = mm::conv(img, w_conv, mm::Border::CONSTANT);

    std::cout << "Imagem original:";            std::cout << mm::drawImg(img) << "\n";
    std::cout << "Kernel original:";            std::cout << mm::drawImg(w) << "\n";
    std::cout << "Kernel rotacionado 180°:";    std::cout << mm::drawImg(w_conv) << "\n";
    std::cout << "Resultado da convolução:";    std::cout << mm::drawImg(conv) << "\n";

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/mm_out_3.cpp -o tmp/mm_out_3 \
  && ./tmp/mm_out_3

La convolución utiliza el *kernel* rotado 180°. Para reproducir la definición matemática de convolución, se rota el *kernel* (aquí, `[[0,1,2],[0,0,0],[0,0,0]]` → `[[0,0,0],[0,0,0],[2,1,0]]`) antes de aplicar `mm::conv`.

### 3.4.4 El Papel del *Kernel*

Los coeficientes del *kernel* determinan completamente el efecto producido por el filtro, tal como se resume en la [Tabela 3.2](#tbl-03-kernels).

<a id="tbl-03-kernels"></a>

**Tabela 3.2:** Interpretación típica de los coeficientes del *kernel*.

| Característica | Efecto típico |
|:---|:---|
| Coeficientes positivos con suma 1 | Suavizado (*pasa-baja*) |
| Suma igual a 0, con valores positivos y negativos | Detección de bordes (*pasa-alta*) |
| Coeficiente central positivo dominante y vecinos negativos | Realce de nitidez |
| Coeficientes asimétricos | Gradiente direccional |


**Ejemplos:**

Suavizado:
$$
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Detección de bordes:
$$
\begin{bmatrix}
-1&-1&-1\\
-1&8&-1\\
-1&-1&-1
\end{bmatrix}
$$

Realce de nitidez:
$$
\begin{bmatrix}
0&-1&0\\
-1&5&-1\\
0&-1&0
\end{bmatrix}
$$

Gradiente direccional (Sobel):
$$
\begin{bmatrix}
-1&0&1\\
-2&0&2\\
-1&0&1
\end{bmatrix}
$$

La [Figura 3.10](#fig-03-convolucao-passo) demuestra el mecanismo paso a paso: para cada posición de la ventana, se multiplica cada coeficiente del *kernel* por el píxel correspondiente de la vecindad y se suman los productos obtenidos. El resultado es exactamente el valor definido por la [Equação 3.11](#eq-03-correlacao) para esa posición de la imagen. Aunque las imágenes producidas por `mm::conv0` y `cv2.filter2D` (o `mm::conv`) sean visualmente muy similares, la implementación basada en OpenCV es miles de veces más rápida, como se muestra a continuación.

> ### ⚠️ Rendimiento: bucles de Python vs. operaciones vectorizadas
>
> La función `mm::conv0` implementa la correlación directamente en Python mediante bucles anidados. Aunque este enfoque es adecuado para fines didácticos, ejecuta un gran número de operaciones y se vuelve lento para imágenes más grandes.
>
> En cambio, `mm::conv` utiliza `cv2.filter2D`, implementado en C++ y optimizado para operaciones matriciales. En el ejemplo presentado, la versión vectorizada fue más de **3000 veces más rápida** que la implementación didáctica, produciendo un resultado visualmente equivalente.
>
> Las diferencias numéricas observadas se concentran principalmente en los bordes de la imagen. En `mm::conv0`, los píxeles del borde permanecen inalterados, mientras que `mm::conv` utiliza una estrategia de reflexión de bordes (`cv2.BORDER_REFLECT_101`, estándar de `cv2.filter2D`).
>
> Por ello, `mm::conv0` debe utilizarse para comprender el algoritmo, mientras que `mm::conv` es la opción recomendada para aplicaciones prácticas.

In [ ]:
%%writefile tmp/fig_03_convolucao_passo.cpp
#define MM_OUT "tmp/fig_03_convolucao_passo.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    //| label: fig-03-convolucao-passo
    //| fig-cap: "Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas."
    //| echo: true
    //| output: true

    mm::Kernel w_mean = mm::Kernel::mean(3);
    mm::Image img_gray = img_leop_gray;

    mm::Image img_conv0 = mm::conv0(img_gray, w_mean);   // laços, bordas preservadas
    mm::Image img_conv  = mm::conv(img_gray, w_mean);    // borda refletida

    std::cout << "Correlacao no pixel central [251,251]:" << "\n";
    std::cout << "  original = " << (int)img_gray.at(251, 251) << "\n";
    std::cout << "  conv0    = " << (int)img_conv0.at(251, 251) << "\n";
    std::cout << "  conv     = " << (int)img_conv.at(251, 251) << "\n";

    mm::show(std::vector<mm::Image>{img_gray, img_conv0, img_conv},
            MM_OUT,
            std::vector<std::string>{"Original", "conv0 (laços)", "conv (vetorizado)"}, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_convolucao_passo_0.png");
mm::write(img_conv0, "tmp/fig_03_convolucao_passo_1.png");
mm::write(img_conv, "tmp/fig_03_convolucao_passo_2.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_convolucao_passo.cpp -o tmp/fig_03_convolucao_passo \
  && ./tmp/fig_03_convolucao_passo \
  && test -f "tmp/fig_03_convolucao_passo.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_convolucao_passo.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_convolucao_passo_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_convolucao_passo_0.png"),
            mm.read("tmp/fig_03_convolucao_passo_1.png"),
            mm.read("tmp/fig_03_convolucao_passo_2.png"),
        ],
        titles=[
            'Original',
            'conv0 (laços)',
            'conv (vetorizado)',
        ],
        cols=3,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.10:** Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas.


In [ ]:
%%writefile tmp/mm_out_4.cpp
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_leop = mm::_read_state("tmp/state/img_leop_12.png");
// [pdi:state-io:end]

    //| echo: false
    // A partir daqui o "sujeito" dos exemplos de filtragem passa a ser o
    // leopardo (mais textura e bordas que o mandril).
    mm::Image img_gray = mm::gray(img_leop);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_45.png");
// [pdi:state-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/mm_out_4.cpp -o tmp/mm_out_4 \
  && ./tmp/mm_out_4

### 3.4.5 Ejemplo Numérico: Correlación Paso a Paso

Para hacer concreto el mecanismo de la [Equação 3.11](#eq-03-correlacao), considere el *kernel* de media 3×3 ($a=b=1$, todos los coeficientes $= 1/9 \approx 0{,}111$) aplicado al *patch* 5×5 extraído de la imagen del leopardo. La [Figura 3.11](#fig-03-patch) muestra el *patch* con la cuadrícula y resalta en amarillo la ventana 3×3 centrada en el píxel $[1,1]$:

In [ ]:
%%writefile tmp/fig_03_patch.cpp
#define MM_OUT "tmp/fig_03_patch.png"
#include "morph.hpp"
#include <iostream>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    //| label: fig-03-patch
    //| fig-cap: "*Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada."
    //| echo: true
    //| output: true

    mm::Image patch = mm::crop(img_gray, 250, 255, 250, 255);
    mm::Kernel B = mm::Kernel::ones(3);

    std::cout << "Patch 5×5 (intensidades):" << std::endl;
    std::cout << mm::drawImg(patch);

    mm::drawImgKernel(patch, B, 1, 1, MM_OUT, 40);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_patch.cpp -o tmp/fig_03_patch \
  && ./tmp/fig_03_patch \
  && test -f "tmp/fig_03_patch.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_patch.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_patch.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_patch.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.11:** *Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada.


Para ilustrar el cálculo de la correlación, considere el píxel en la posición $[1,1]$ del *patch* 5×5 mostrado en la [Figura 3.11](#fig-03-patch).. Esa posición fue elegida solo por conveniencia didáctica, ya que posee una vecindad 3×3 completa a su alrededor.

Los valores de esa vecindad corresponden a la submatriz superior izquierda del *patch*:

$$
\text{vecindad} = \begin{bmatrix}
 91 &  95 & 108 \\
106 & 107 & 108 \\
102 & 103 & 107
\end{bmatrix}
$$

Aplicando la [Equação 3.11](#eq-03-correlacao) con el kernel de media:

$$
g[1,1] = \frac{ 91+95+108+106+107+108+102+103+107}{9} = \frac{927}{9} = 103
$$

El resultado (103) es ligeramente menor que el valor original del píxel central (107), pues la media incorpora vecinos de menor intensidad, produciendo el efecto de suavizado. En la práctica, el algoritmo inicia el procesamiento en $[0,0]$ y repite ese mismo cálculo para cada posición de la imagen, desplazando la ventana hasta cubrir todo el dominio.

## 3.5 Filtrado Espacial de Suavizado

Los filtros de suavizado (*smoothing filters*) atenúan variaciones bruscas de intensidad, reduciendo ruido y detalles de alta frecuencia. Son filtros **paso-bajo** — preservan las componentes de baja frecuencia (estructuras grandes) y atenúan las de alta frecuencia (ruido, bordes).

### 3.5.1 Filtro de Media (*Box Filter*)

El filtro de media utiliza un *kernel* uniforme de tamaño $n \times n$, donde todos los coeficientes valen $1/n^2$:

<a id="eq-03-media"></a>
$$
w_{\text{media}} = \frac{1}{n^2}
\begin{bmatrix}
1 & \cdots & 1 \\
\vdots & \ddots & \vdots \\
1 & \cdots & 1
\end{bmatrix}_{n \times n} \tag{3.13}
$$


Cada píxel de salida es la media aritmética de los $n^2$ píxeles de su vecindario. Nótese que la suma de los coeficientes es siempre 1 — el brillo medio de la imagen se preserva. Los *kernels* más grandes producen un suavizado más agresivo, pero desenfocan progresivamente los bordes.

[Figura 3.12](#fig-03-media) muestra el efecto del filtro de media con *kernels* $3\times3$, $7\times7$ y $15\times15$ sobre un detalle de la imagen del leopardo. Los resultados se obtuvieron con `mm::blur`, que implementa el filtro de media mediante la función `cv2.blur`, equivalente a la convolución de la imagen con un *kernel* uniforme cuyos coeficientes son $h(x,y)=1/N^2$; de forma equivalente, el mismo resultado puede obtenerse con `mm::conv`, calculando ($g=f*h$). A medida que el *kernel* aumenta, más píxeles contribuyen a cada valor de salida, intensificando el suavizado, reduciendo el ruido y haciendo que los detalles finos y los bordes se vuelvan progresivamente más borrosos.

In [ ]:
%%writefile tmp/fig_03_media.cpp
#define MM_OUT "tmp/fig_03_media.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // Detalhe da região do olho
    int y0 = 580, y1 = 740, x0 = 680, x1 = 900;
    mm::Image img_gray_crop = mm::crop(img_gray, y0, y1, x0, x1);

    std::vector<int> sizes = {3, 7, 15};
    std::vector<mm::Image> imgs;
    imgs.push_back(img_gray_crop);
    for (int k : sizes) {
        imgs.push_back(mm::blur(img_gray_crop, k)); // ou
        // mm::Kernel kernel = mm::Kernel::mean(k);
        // imgs.push_back(mm::conv(img_gray_crop, kernel));
    }
    std::vector<std::string> titles = {"Original"};
    for (int k : sizes) {
        titles.push_back("Média " + std::to_string(k) + "×" + std::to_string(k));
    }

    mm::show(imgs, MM_OUT, titles, 4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_media.cpp -o tmp/fig_03_media \
  && ./tmp/fig_03_media \
  && test -f "tmp/fig_03_media.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_media.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_media.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_media.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.12:** Filtro de média com *kernels* de tamanho crescente (3×3, 7×7, 15×15). O borramento das bordas aumenta com o tamanho do *kernel*.


### 3.5.2 Filtro Gaussiano

El filtro Gaussiano pondera los píxeles de la vecindad de acuerdo con una función Gaussiana bidimensional:

<a id="eq-03-gaussiana"></a>
$$
G(s,t) = \frac{1}{2\pi\sigma^2}\,e^{-\frac{s^2+t^2}{2\sigma^2}} \tag{3.14}
$$


donde $\sigma$ es la desviación estándar y controla el radio de influencia. Los píxeles más cercanos al centro tienen un peso mayor; los píxeles distantes se ignoran progresivamente.

La [Figura 3.13](#fig-03-gauss-kernel) presenta el *kernel* Gaussiano $5\times5$ generado para $\sigma=1$. El *kernel* fue construido a partir del producto externo de dos vectores Gaussianos unidimensionales y posteriormente normalizado para que la suma de sus coeficientes sea igual a $1$. Se observa que los mayores pesos se concentran en el centro de la matriz, decreciendo radialmente hacia los bordes. Esta distribución hace que los píxeles centrales tengan mayor influencia en el resultado del filtrado, contribuyendo a una suavización más natural y con mejor preservación de bordes que el filtro de media.

In [ ]:
%%writefile tmp/fig_03_gauss_kernel.cpp
#define MM_OUT "tmp/fig_03_gauss_kernel.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <numeric>
#include <iomanip>

int main() {
    //| label: fig-03-gauss-kernel
    //| fig-cap: "*Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente."
    //| echo: true
    //| output: true

    mm::Kernel w = mm::Kernel::gaussian(5, 1.0);   // mm::Kernel::gaussian(5, 1.0) na morph.hpp

    std::cout << "Kernel Gaussiano 5x5 (s=1), normalizado:" << "\n";
    for (int y = 0; y < 5; y++) {
        std::cout << "  ";
        for (int x = 0; x < 5; x++) {
            std::cout << std::fixed << std::setprecision(4) << w.at(y, x);
            if (x < 4) std::cout << "  ";
        }
        std::cout << "\n";
    }
    std::cout << "Peso central [2,2] = " << std::fixed << std::setprecision(4) << w.at(2, 2) << "   |   canto [0,0] = " << w.at(0, 0) << "\n";

    // Visualização: resposta do filtro Gaussiano a um impulso central
    mm::Image impulso(5, 5);
    impulso.at(2, 2) = 255;
    mm::drawImgPlt(mm::gaussian(impulso, 5, 1.0), MM_OUT);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_gauss_kernel.cpp -o tmp/fig_03_gauss_kernel \
  && ./tmp/fig_03_gauss_kernel \
  && test -f "tmp/fig_03_gauss_kernel.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_gauss_kernel.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_gauss_kernel.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_gauss_kernel.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.13:** *Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente.


> ### 📝 Ventaja computacional de la separabilidad
>
> Considere un *kernel* cuadrado de tamaño $n \times n$. Si este filtro es separable (como el Gaussiano), la convolución 2D puede descomponerse en dos convoluciones 1D: una horizontal y otra vertical.
>
> En ese caso, el costo por píxel pasa de aproximadamente $O(n^2)$ operaciones (convolución 2D directa) a $O(2n)$ operaciones (dos convoluciones 1D). Así, la complejidad se reduce de forma significativa, haciendo el procesamiento más eficiente.

En comparación con el filtro de media, el Gaussiano:

- **Preserva mejor los bordes** — la ponderación radial suaviza sin crear transiciones abruptas;
- **No introduce anillos** (*ringing*) en el dominio de la frecuencia, pues la Gaussiana es su propia transformada de Fourier (Capítulo 5);
- **Está controlado por $\sigma$** — aumentar $\sigma$ equivale a aumentar el radio de suavizado de forma continua y predecible.

La [Figura 3.14](#fig-03-gauss) compara los filtros de media y gaussiano aplicados a la imagen del leopardo usando una ventana $9\times9$. El filtro de media se implementó mediante convolución con un *kernel* uniforme, donde todos los $81$ píxeles de la vecindad poseen el mismo peso ($1/81$), mientras que el filtro gaussiano se obtuvo con `cv2.GaussianBlur`, utilizando pesos definidos por una distribución gaussiana. Ambos reducen ruido y suavizan la imagen, pero el filtro gaussiano preserva mejor los bordes y los detalles locales, como puede observarse en la región ampliada del ojo.

La [Figura 3.14](#fig-03-gauss) compara los filtros de media y gaussiano aplicados a un detalle de la imagen del leopardo con *kernels* $9\times 9$. El filtro de media se obtuvo con `mm::blur`, equivalente a la convolución con un *kernel* uniforme cuyos coeficientes valen $1/81$, mientras que el filtro gaussiano se obtuvo con `mm::gaussian`, equivalente a la convolución con un *kernel* generado a partir de una distribución gaussiana. Ambos promueven suavización y reducción de ruido, pero el filtro gaussiano asigna mayor peso a los píxeles centrales de la vecindad, preservando mejor los bordes y los detalles locales, como puede observarse en la región ampliada del ojo.

In [ ]:
%%writefile tmp/fig_03_gauss.cpp
#define MM_OUT "tmp/fig_03_gauss.png"
// Compile: g++ -std=c++17 -O2 -o prog prog.cpp -lm && ./prog
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

//| label: fig-03-gauss
//| fig-cap: "Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // img_gray é fornecido automaticamente (não declarar)

    mm::Image img_media9 = mm::blur(img_gray, 9);
    mm::Image img_gauss9 = mm::gaussian(img_gray, 9, 0);

    // Detalhe da região do olho
    mm::Image img_gray_crop   = mm::crop(img_gray,    580, 740, 680, 900);
    mm::Image img_media9_crop = mm::crop(img_media9,  580, 740, 680, 900);
    mm::Image img_gauss9_crop = mm::crop(img_gauss9,  580, 740, 680, 900);

    mm::show(
        std::vector<mm::Image>{img_gray_crop, img_media9_crop, img_gauss9_crop},
        MM_OUT,
        std::vector<std::string>{"Detalhe: Original", "Média 9×9", "Gaussiano 9×9"},
        3
    );

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray_crop, "tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_gauss_0.png");
mm::write(img_media9_crop, "tmp/fig_03_gauss_1.png");
mm::write(img_gauss9_crop, "tmp/fig_03_gauss_2.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_gauss.cpp -o tmp/fig_03_gauss \
  && ./tmp/fig_03_gauss \
  && test -f "tmp/fig_03_gauss.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_gauss.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_gauss_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_gauss_0.png"),
            mm.read("tmp/fig_03_gauss_1.png"),
            mm.read("tmp/fig_03_gauss_2.png"),
        ],
        titles=[
            'Detalhe: Original',
            'Média 9×9',
            'Gaussiano 9×9',
        ],
        cols=3,
        figsize=(12, 8),
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.14:** Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto.


## 3.6 Filtrado Espacial de Realce

Los filtros de realce (*sharpening filters*) enfatizan transiciones abruptas de intensidad, aumentando la nitidez y la visibilidad de bordes. Son filtros **paso-alto** — amplifican los componentes de alta frecuencia (bordes, textura) y suprimen los de baja frecuencia (regiones uniformes).

La intuición es simple: si restamos de una imagen su versión suavizada (que contiene solo las bajas frecuencias), lo que queda son las altas frecuencias — bordes y detalles. Sumando ese residuo de vuelta a la imagen original, el contraste local aumenta:

<a id="eq-03-realce-intuitivo"></a>
$$
g = f + k\,(f - f_{\text{suave}}), \quad k > 0 \tag{3.15}
$$


La [Figura 3.15](#fig-03-sim-03-filtragem1d) ilustra este proceso en una señal 1D sintética con tres estructuras distintas: un escalón ancho, un pico fino y una rampa suave. En el panel ①, la señal original $f(x)$; en el ②, la versión suavizada $f_{\text{suave}}(x)$ obtenida por media móvil — nótese cómo el pico fino se atenúa. El panel ③ muestra el residuo $f - f_{\text{suave}}$, que retiene solo las transiciones abruptas. Finalmente, el panel ④ muestra $g(x)$: el pico, antes atenuado, se restaura y amplifica en relación con el original. Ajuste $k$ y el tamaño de la ventana para observar el *trade-off* entre nitidez y amplificación de ruido.

Los filtros de realce formalizan esta idea directamente en el *kernel*, sin necesidad de dos etapas separadas.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-03-filtragem1d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎮 Simulador: Filtrado Espacial de Realce 1D</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k·(f − f_suave)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">

    <!-- Controles -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;flex-wrap:wrap;gap:16px;align-items:flex-end;">
      
      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#b9770e;">Parámetro k (Realce)</label>
        <input id="sim_cap03_k_slider" type="range" min="0" max="5" step="0.1" value="1.5" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">k = <span id="sim_cap03_k_val">1.5</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">Ventana de Suavizado (pts)</label>
        <input id="sim_cap03_win_slider" type="range" min="3" max="31" step="2" value="9" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">ventana = <span id="sim_cap03_win_val">9</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">Nivel de Ruido σ</label>
        <input id="sim_cap03_noise_slider" type="range" min="0" max="0.3" step="0.01" value="0.04" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">σ = <span id="sim_cap03_noise_val">0.04</span></span>
      </div>

    </div>

    <!-- Gráficos com Fundos Pastéis -->
    <div style="display:flex;flex-direction:column;gap:6px;">

      <!-- ① Azul Pastéis -->
      <div style="background:#ebf4fd;border:1px solid #d4e5f7;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#1a5fa8;margin-bottom:3px;font-family:monospace;">① f(x) — Señal Original</div>
        <canvas id="sim_cap03_c1" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ filtro pasa-bajas (media móvil)</div>

      <!-- ② Verde Pastéis -->
      <div style="background:#eaf7f2;border:1px solid #d1efe3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#0d6b4f;margin-bottom:3px;font-family:monospace;">② f_suave(x) — Pico Atenuado por el Filtro</div>
        <canvas id="sim_cap03_c2" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ sustracción: f − f_suave</div>

      <!-- ③ Salmão/Rosa Pastéis -->
      <div style="background:#fef0eb;border:1px solid #fcdad0;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#a03010;margin-bottom:3px;font-family:monospace;">③ Residuo (f − f_suave) — Altas Frecuencias / Bordes</div>
        <canvas id="sim_cap03_c3" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ suma: f + k · residuo</div>

      <!-- ④ Roxo Pastéis -->
      <div style="background:#f2f0fd;border:1px solid #e1dcf9;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#4a3faa;margin-bottom:3px;font-family:monospace;">④ g(x) — Señal con Pico Realzado</div>
        <canvas id="sim_cap03_c4" height="80" style="width:100%;display:block;"></canvas>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimCap03(root){
    if (!root || root.dataset.simCap03Init) return;
    root.dataset.simCap03Init = "1";

    var N = 250;
    var peak_i = 115;

    var slK     = root.querySelector('#sim_cap03_k_slider');
    var slWin   = root.querySelector('#sim_cap03_win_slider');
    var slNoise = root.querySelector('#sim_cap03_noise_slider');

    var valK     = root.querySelector('#sim_cap03_k_val');
    var valWin   = root.querySelector('#sim_cap03_win_val');
    var valNoise = root.querySelector('#sim_cap03_noise_val');

    function make_signal(noise) {
      var seed = 42;
      function rand() { seed = (seed * 9301 + 49297) % 233280; return seed / 233280; }
      function randn() { return Math.sqrt(-2 * Math.log(rand() + 1e-9)) * Math.cos(2 * Math.PI * rand()); }
      var f = [];
      for (var i = 0; i < N; i++) {
        var v = 0.2;
        if (i >= 30  && i <= 80)  v += 0.7;
        if (i >= 110 && i <= 120) v += 1.0;
        if (i >= 150 && i <= 190) v += 0.5 * (i - 150) / 40;
        v += noise * randn();
        f.push(v);
      }
      return f;
    }

    function moving_avg(f, win) {
      var half = Math.floor(win / 2);
      return f.map(function(_, i){
        var s = 0, c = 0;
        for (var j = Math.max(0, i - half); j <= Math.min(f.length - 1, i + half); j++) {
          s += f[j];
          c++;
        }
        return s / c;
      });
    }

    function drawCanvas(canvasId, datasets, annotations) {
      var canvas = root.querySelector('#' + canvasId);
      if (!canvas) return;
      canvas.width = canvas.offsetWidth || 800;
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var padL = 8, padR = 8, padT = 6, padB = 6;
      var allVals = [].concat.apply([], datasets.map(function(d){ return d.data; }));
      var mn = Math.min.apply(null, allVals);
      var mx = Math.max.apply(null, allVals);
      var rng = mx - mn || 1;
      var drawH = H - padT - padB, drawW = W - padL - padR;

      function toY(v){ return H - padB - (v - mn) / rng * drawH; }
      function toX(i){ return padL + i / (N - 1) * drawW; }

      // Faixa de destaque do pico
      var xA = toX(108), xB = toX(122);
      ctx.fillStyle = 'rgba(255, 200, 50, 0.15)';
      ctx.fillRect(xA, padT, xB - xA, drawH);

      // Grade
      ctx.strokeStyle = 'rgba(38, 36, 29, 0.08)';
      ctx.lineWidth = 0.5;
      for (var g = 0; g <= 3; g++) {
        var gy = padT + g / 3 * drawH;
        ctx.beginPath(); ctx.moveTo(padL, gy); ctx.lineTo(W - padR, gy); ctx.stroke();
      }

      // Linha do zero
      if (mn < 0 && mx > 0) {
        ctx.strokeStyle = 'rgba(38, 36, 29, 0.2)';
        ctx.lineWidth = 0.8;
        ctx.setLineDash([4, 3]);
        var y0 = toY(0);
        ctx.beginPath(); ctx.moveTo(padL, y0); ctx.lineTo(W - padR, y0); ctx.stroke();
        ctx.setLineDash([]);
      }

      // Desenho das séries
      datasets.forEach(function(ds){
        ctx.strokeStyle = ds.color;
        ctx.lineWidth   = ds.width || 1.8;
        ctx.globalAlpha = ds.alpha || 1;
        if (ds.dash) ctx.setLineDash(ds.dash); else ctx.setLineDash([]);
        ctx.beginPath();
        ds.data.forEach(function(v, i){
          if (i === 0) ctx.moveTo(toX(i), toY(v)); else ctx.lineTo(toX(i), toY(v));
        });
        ctx.stroke();
        ctx.setLineDash([]);
        ctx.globalAlpha = 1;
      });

      // Anotações
      if (annotations) {
        annotations.forEach(function(an){
          var xi = toX(an.i), yi = toY(an.v);
          ctx.strokeStyle = an.color || '#5e5a4a';
          ctx.lineWidth = 1;
          ctx.setLineDash([3, 3]);
          ctx.beginPath(); ctx.moveTo(xi, padT); ctx.lineTo(xi, H - padB); ctx.stroke();
          ctx.setLineDash([]);

          ctx.fillStyle = an.color || '#5e5a4a';
          ctx.beginPath(); ctx.arc(xi, yi, 4, 0, 2 * Math.PI); ctx.fill();

          ctx.fillStyle = an.color || '#26241d';
          ctx.font = 'bold 10px monospace';
          ctx.fillText(an.label, xi + 6, Math.max(padT + 12, Math.min(H - padB - 4, yi - 6)));
        });
      }
    }

    function update() {
      var k     = parseFloat(slK.value) || 0;
      var win   = parseInt(slWin.value) || 3;
      var noise = parseFloat(slNoise.value) || 0;

      valK.textContent     = k.toFixed(1);
      valWin.textContent   = win;
      valNoise.textContent = noise.toFixed(2);

      var f       = make_signal(noise);
      var f_suave = moving_avg(f, win);
      var residuo = f.map(function(v, i){ return v - f_suave[i]; });
      var g       = f.map(function(v, i){ return v + k * residuo[i]; });

      var pk = peak_i;
      var peakF = f[pk], peakS = f_suave[pk], peakR = residuo[pk], peakG = g[pk];

      // ① Original (Azul)
      drawCanvas('sim_cap03_c1',
        [{ data: f, color: '#1a5fa8' }],
        [{ i: pk, v: peakF, color: '#1a5fa8', label: 'pico' }]
      );

      // ② Suavizado (Verde)
      drawCanvas('sim_cap03_c2',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: f_suave, color: '#0d6b4f', width: 2 }],
        [{ i: pk, v: peakS, color: '#0d6b4f', label: 'atenuado' }]
      );

      // ③ Resíduo (Salmão / Laranja)
      drawCanvas('sim_cap03_c3',
        [{ data: residuo, color: '#a03010' }],
        [{ i: pk, v: peakR, color: '#a03010', label: 'borda detectada' }]
      );

      // ④ Realçado (Roxo)
      drawCanvas('sim_cap03_c4',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: g, color: '#4a3faa', width: 2.2 }],
        [{ i: pk, v: peakG, color: '#4a3faa', label: 'pico realçado' }]
      );
    }

    [slK, slWin, slNoise].forEach(function(sl){
      sl.addEventListener('input', update);
    });

    update();
  }

  function tryInitSimCap03(){
    var root = document.getElementById('sim-03-filtragem1d');
    if (root) initSimCap03(root); else setTimeout(tryInitSimCap03, 200);
  }
  tryInitSimCap03();
})();
</script>
</div>
""")

**Figura 3.15:** Simulador: Filtrado Espacial de Realce 1D (Unsharp Masking y High-Boost)


<figure id="fig-03-sim-03-filtragem1d">
  <img src="imagens/fig-03-sim-03-filtragem1d.png" alt=" Simulador: Filtrado Espacial de Realce 1D (Unsharp Masking y High-Boost) " style="max-width:80%" />
  <figcaption><strong>Figura 3.15:</strong>  Simulador: Filtrado Espacial de Realce 1D (Unsharp Masking y High-Boost) </figcaption>
</figure>

### 3.6.1 Laplaciano

El Laplaciano es un operador de **segunda derivada** isotrópico, es decir, responde de igual manera a las variaciones en todas las direcciones, a diferencia de los operadores de primera derivada, como Sobel y Prewitt, que son direccionales:

<a id="eq-03-laplaciano"></a>
$$
\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2} \tag{3.16}
$$


Una propiedad importante de la segunda derivada es que su valor es **cercano a cero en regiones uniformes** y elevado en las transiciones de intensidad. Así, al restar el Laplaciano de la imagen original, se refuerzan los bordes y detalles, aumentando el contraste local:

<a id="eq-03-realce-lap"></a>
$$
g(x,y) = f(x,y) - \nabla^2 f(x,y) \tag{3.17}
$$


En la forma discreta, la segunda derivada en $x$ se aproxima por $f(x+1,y) - 2f(x,y) + f(x-1,y)$, y de manera análoga en $y$. Sumando ambas direcciones, se obtiene el *kernel* $w_4$ (4-vecinos) o $w_8$ (8-vecinos, incluyendo diagonales):

<a id="eq-03-laplaciano-kernel"></a>
$$
w_4 = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}, \qquad
w_8 = \begin{bmatrix} 1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1 \end{bmatrix} \tag{3.18}
$$


> ### 📝 Suma cero y centro negativo
>
> Ambos *kernels* tienen **suma de coeficientes igual a cero**: en regiones uniformes, la salida es 0 — el Laplaciano no altera el brillo medio, solo detecta variaciones. El **centro negativo** indica que el píxel se compara con sus vecinos: cuanto más se destaque (hacia arriba o hacia abajo), mayor será el valor absoluto del Laplaciano en ese punto.

En el siguiente ejemplo, el píxel central $[1,1]=107$ posee vecinos $\{95, 106, 108, 103\}$. Como estos valores son cercanos entre sí, la región es casi uniforme y el Laplaciano devuelve un valor bajo, produciendo poco realce. En regiones de borde, donde hay diferencias mayores entre el píxel central y sus vecinos, el Laplaciano asume valores más elevados (positivos o negativos), y la operación de [Equação 3.17](#eq-03-realce-lap) intensifica esas transiciones.

La [Figura 3.16](#fig-03-laplaciano-patch) ilustra el cálculo del Laplaciano con el *kernel* $w_4$ en una vecindad $3\times3$ destacada dentro de un *patch* $5\times5$. El ejemplo muestra el valor obtenido por el operador y el correspondiente píxel realzado en la imagen de salida, que pasa de 107 a 123.

In [ ]:
%%writefile tmp/fig_03_laplaciano_patch.cpp
#define MM_OUT "tmp/fig_03_laplaciano_patch.png"
// Compile with: g++ -std=c++17 -o program program.cpp -lmorph
#include "morph.hpp"
#include <iostream>

//| label: fig-03-laplaciano-patch
//| fig-cap: "*Kernel* Laplaciano w4 sobre el *patch* 5×5: la ventana amarilla destaca la vecindad 3×3 donde el operador de segunda derivada se calcula."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    mm::Image patch = mm::crop(img_gray, 250, 255, 250, 255);
    mm::Kernel B = mm::Kernel::ones(3);

    std::cout << "Patch 5x5 (intensidades):\n";
    std::cout << mm::drawImg(patch) << "\n";

    mm::drawImgKernel(patch, B, 1, 1, MM_OUT, 40);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_laplaciano_patch.cpp -o tmp/fig_03_laplaciano_patch \
  && ./tmp/fig_03_laplaciano_patch \
  && test -f "tmp/fig_03_laplaciano_patch.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_laplaciano_patch.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_laplaciano_patch.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_laplaciano_patch.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.16:** *Kernel* Laplaciano w4 sobre o *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado.


A [Figura 3.17](#fig-03-laplaciano) compara a aplicação de los *kernels* laplacianos $w_4$ y $w_8$ en un recorte más grande de la imagen del leopardo. Para cada caso, se muestran la respuesta bruta del operador, que evidencia los bordes y las transiciones de intensidad, y la imagen obtenida tras el realce por sustracción del laplaciano. Se observa que el *kernel* $w_8$, al considerar también los vecinos diagonales, produce una respuesta más intensa y detecta variaciones en más direcciones, resultando en un realce ligeramente más acentuado.

In [ ]:
%%writefile tmp/fig_03_laplaciano.cpp
#define MM_OUT "tmp/fig_03_laplaciano.png"
//| label: fig-03-laplaciano
//| fig-cap: "Laplaciano aplicado à imagem do leopardo: resposta bruta (bordas) com w4 e w8, e imagens realçadas pela subtração do Laplaciano. w8 é mais sensível às diagonais."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // img_gray_crop is provided automatically

    mm::Kernel w4{{0,1,0},{1,-4,1},{0,1,0}};
    mm::Kernel w8{{1,1,1},{1,-8,1},{1,1,1}};

    mm::show(
        {img_gray_crop, mm::laplacian_viz(img_gray_crop, w4), mm::laplacian(img_gray_crop, w4),
         img_gray_crop, mm::laplacian_viz(img_gray_crop, w8), mm::laplacian(img_gray_crop, w8)},
        MM_OUT,
        {"Original", "Laplaciano w4", "Realce w4",
         "Original", "Laplaciano w8", "Realce w8"},
        3
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_laplaciano.cpp -o tmp/fig_03_laplaciano \
  && ./tmp/fig_03_laplaciano \
  && test -f "tmp/fig_03_laplaciano.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_laplaciano.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_laplaciano.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_laplaciano.png"), figsize=(14, 8))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.17:** Laplaciano aplicado à imagem do leopardo: resposta bruta (bordas) com w4 e w8, e imagens realçadas pela subtração do Laplaciano. w8 é mais sensível às diagonais.


### 3.6.2 Operador de Sobel

El operador de Sobel estima las **derivadas parciales de primer orden** en las direcciones horizontal y vertical. A diferencia del Laplaciano (segunda derivada), el Sobel es direccional y más robusto al ruido, pues cada *kernel* combina una derivada con un suavizado Gaussiano perpendicular:

<a id="eq-03-sobel"></a>
$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * f \tag{3.19}
$$


$G_x$ detecta bordes **verticales** (variación en la dirección $x$); $G_y$ detecta bordes **horizontales** (variación en la dirección $y$). Los pesos $\{1,2,1\}$ en la dirección perpendicular corresponden al suavizado Gaussiano 1D, que reduce la sensibilidad al ruido.

> ### 📝 Sobel es correlación, no convolución
>
> Los *kernels* de Sobel son **asimétricos** — la rotación de 180° altera el resultado. `cv2.Sobel` implementa correlación cruzada (como `cv2.filter2D`). Para obtener la derivada direccional correcta, las señales ya están definidas para correlación: $G_x$ devuelve valores positivos donde la intensidad crece de izquierda a derecha.

La magnitud del **gradiente** combina los dos componentes, representando la fuerza del borde independientemente de la dirección:

<a id="eq-03-gradiente"></a>
$$
|\nabla f| = \sqrt{G_x^2 + G_y^2} \tag{3.20}
$$


Y la **dirección** del gradiente (perpendicular al borde) es:

<a id="eq-03-direcao"></a>
$$
\theta = \arctan\left(\frac{G_y}{G_x}\right) \tag{3.21}
$$


Para ilustrar numéricamente, se calculan $G_x$ y $G_y$ manualmente en el píxel central $[1,1]$ del *patch* de 5×5:

El valor reducido de $|{\nabla f}|$ en ese *patch* confirma que la región es casi uniforme, pues el gradiente asume valores elevados solo donde hay cambios significativos de intensidad. La [Figura 3.18](#fig-03-sobel) aplica el operador de Sobel a un recorte mayor de la imagen del leopardo. Se presentan las respuestas horizontal ($G_x$) y vertical ($G_y$), obtenidas por convolución con los respectivos *kernels* de Sobel, además de la magnitud $|{\nabla f}|$, calculada a partir de la combinación de ambas. Mientras que $G_x$ destaca bordes verticales y $G_y$ bordes horizontales, la magnitud evidencia bordes en cualquier dirección.

In [ ]:
%%writefile tmp/fig_03_sobel.cpp
#define MM_OUT "tmp/fig_03_sobel.png"
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-sobel
    //| fig-cap: "Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os gradientes horizontal e vertical, revelando todas as bordas. A decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve a magnitude já com clip."
    //| echo: true
    //| output: true

    mm::Image mag = mm::sobel(img_gray_crop);

    mm::show(std::vector<mm::Image>{img_gray_crop, mag},
             MM_OUT,
             std::vector<std::string>{"Original", "Magnitude |grad f| (mm.sobel)"}, 2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_sobel_0.png");
mm::write(mag, "tmp/fig_03_sobel_1.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_sobel.cpp -o tmp/fig_03_sobel \
  && ./tmp/fig_03_sobel \
  && test -f "tmp/fig_03_sobel.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_sobel.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_sobel_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_sobel_0.png"),
            mm.read("tmp/fig_03_sobel_1.png"),
        ],
        titles=[
            'Original',
            'Magnitude |grad f| (mm.sobel)',
        ],
        cols=2,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.18:** Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os gradientes horizontal e vertical, revelando todas as bordas. A decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve a magnitude já com clip.


### 3.6.3 Operador de Prewitt

El operador de Prewitt es estructuralmente idéntico al de Sobel, pero sustituye la ponderación gaussiana $\{1,2,1\}$ por pesos uniformes $\{1,1,1\}$:

<a id="eq-03-prewitt"></a>
$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -1 & -1 \\ 0 & 0 & 0 \\ 1 & 1 & 1 \end{bmatrix} * f \tag{3.22}
$$


La magnitud y la dirección del gradiente siguen las mismas ecuaciones que las de Sobel ([Equação 3.20](#eq-03-gradiente) y [Equação 3.21](#eq-03-direcao)). La diferencia práctica es que Prewitt es ligeramente más sensible al ruido — el suavizado perpendicular uniforme pondera menos el píxel central de la línea — pero computacionalmente más simple. En imágenes con bajo ruido los resultados son equivalentes.

In [ ]:
%%writefile tmp/fig_03_prewitt.cpp
#define MM_OUT "tmp/fig_03_prewitt.png"
// Compile with: g++ -std=c++17 -o program program.cpp morph.hpp
#include "morph.hpp"
#include <iostream>
#include <vector>

//| label: fig-03-prewitt
//| fig-cap: "Operador de Prewitt: magnitude do gradiente, comparável ao Sobel mas sem a ponderação central. mm::prewitt devolve |∇f| com clip."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::show(std::vector<mm::Image>{img_gray_crop, mm::prewitt(img_gray_crop)},
             MM_OUT,
             std::vector<std::string>{"Original", "Magnitude |grad f| (mm.prewitt)"}, 2);
    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_prewitt.cpp -o tmp/fig_03_prewitt \
  && ./tmp/fig_03_prewitt \
  && test -f "tmp/fig_03_prewitt.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_prewitt.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_prewitt.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_prewitt.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.19:** Operador de Prewitt: magnitude do gradiente, comparável ao Sobel mas sem a ponderação central. mm::prewitt devolve |∇f| com clip.


### 3.6.4 *Unsharp Masking* (USM)

El *Unsharp Masking* es una técnica clásica de realce de nitidez originaria de la fotografía analógica, hoy ampliamente utilizada en software de edición de imágenes. La idea central es extraer los **componentes de alta frecuencia** de la imagen (bordes y detalles) y sumarlos de vuelta a la original con un peso $k$:

<a id="tbl-03-usm"></a>

**Tabela 3.3:** Etapas del *Unsharp Masking*.

| Etapa | Operación | Descripción |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * G_\sigma$ | Suaviza con gaussiana — retiene bajas frecuencias |
| 2 | $m = f - \bar{f}$ | Máscara: diferencia = altas frecuencias (bordes) |
| 3 | $g = f + k \cdot m$ | Suma ponderada de la máscara a la original |


Sustituyendo la etapa 2 en la etapa 3, se obtiene la expresión compacta:

<a id="eq-03-usm"></a>
$$
g = f + k\,(f - f*G_\sigma) = (1+k)\,f - k\,(f*G_\sigma) \tag{3.23}
$$


El parámetro $k$ controla la intensidad del realce:

- $k = 0$: sin realce ($g = f$);
- $k = 1$: USM clásico — duplica la contribución de las altas frecuencias;
- $k > 1$: *High Boost Filtering* — amplificación más allá del doble, útil para imágenes muy borrosas.

> ### ⚠️ Amplificación de ruido
>
> El USM no distingue bordes de ruido — ambos son componentes de alta frecuencia. Para $k$ elevado, el ruido presente en la imagen se amplifica junto con los bordes. Por ello, se recomienda aplicar una leve suavización antes del USM en imágenes ruidosas, o usar un $\sigma$ pequeño en la gaussiana.

Para ilustrar las etapas del USM, la [Figura 3.20](#fig-03-usm2) aplica el método a un *parche* $30\times30$ de la imagen del leopardo, utilizando $\sigma=1$ y $k=1$. Inicialmente, la imagen se suaviza mediante un filtro gaussiano. A continuación, la máscara de alta frecuencia se obtiene como la diferencia entre la imagen original y la suavizada. Finalmente, esta máscara se suma a la imagen original, reforzando bordes y detalles. La figura presenta las tres etapas del proceso y el resultado final del realce.

In [ ]:
%%writefile tmp/fig_03_usm2.cpp
#define MM_OUT "tmp/fig_03_usm2.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-usm2
    //| fig-cap: "Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada."
    //| echo: true
    //| output: true

    mm::Image patch = mm::crop(img_gray_crop, 35, 65, 45, 75);

    mm::Image p_suave = mm::gaussian(patch, 7, 1.0);   // suavização Gaussiana
    mm::Image p_usm   = mm::usm(patch, 1.0);           // realce completo (k = 1.0)

    mm::show(std::vector<mm::Image>{patch, p_suave, p_usm},
             MM_OUT,
             std::vector<std::string>{"Patch original", "Suavizado (σ=1)", "Realçado USM (k=1)"}, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(patch, "tmp/fig_03_usm2_0.png");
mm::write(p_suave, "tmp/fig_03_usm2_1.png");
mm::write(p_usm, "tmp/fig_03_usm2_2.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_usm2.cpp -o tmp/fig_03_usm2 \
  && ./tmp/fig_03_usm2 \
  && test -f "tmp/fig_03_usm2.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_usm2.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_usm2_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_usm2_0.png"),
            mm.read("tmp/fig_03_usm2_1.png"),
            mm.read("tmp/fig_03_usm2_2.png"),
        ],
        titles=[
            'Patch original',
            'Suavizado (σ=1)',
            'Realçado USM (k=1)',
        ],
        cols=3,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.20:** Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada.


La [Figura 3.21](#fig-03-usm) aplica el método USM a un recorte más grande de la imagen del leopardo utilizando $\sigma=1$ y diferentes valores del factor de ganancia $k$. En todos los casos, la máscara de alta frecuencia se obtiene mediante la diferencia entre la imagen original y su versión suavizada por filtro gaussiano. El parámetro $k$ controla la intensidad del realce: valores menores producen un aumento sutil de nitidez, mientras que valores mayores refuerzan progresivamente bordes y detalles. Se observa que, para valores elevados de $k$, surgen halos alrededor de los bordes y el ruido presente en la imagen comienza a amplificarse.

In [ ]:
%%writefile tmp/fig_03_usm.cpp
#define MM_OUT "tmp/fig_03_usm.png"
// Compile with: g++ -std=c++17 -o program program.cpp -lmorph
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-usm
    //| fig-cap: "*Unsharp Masking* na imagem do leopardo com σ=1 e k de 0.5 a 8.0. Para k>2 surgem halos nas bordas e o ruído de fundo aparece."
    //| echo: true
    //| output: true

    mm::show(
        std::vector<mm::Image>{img_gray_crop,
             mm::usm(img_gray_crop, 0.5), mm::usm(img_gray_crop, 1.0),
             mm::usm(img_gray_crop, 3.0), mm::usm(img_gray_crop, 5.0),
             mm::usm(img_gray_crop, 8.0)},
        MM_OUT,
        std::vector<std::string>{"Original", "USM k=0.5", "USM k=1.0", "USM k=3.0", "USM k=5.0", "USM k=8.0"},
        3
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_usm.cpp -o tmp/fig_03_usm \
  && ./tmp/fig_03_usm \
  && test -f "tmp/fig_03_usm.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_usm.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_usm.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_usm.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.21:** *Unsharp Masking* na imagem do leopardo com σ=1 e k de 0.5 a 8.0. Para k>2 surgem halos nas bordas e o ruído de fundo aparece.


### 3.6.5 Detector de Canny

Canny combina cuatro etapas en secuencia — suavizado Gaussiano, gradiente de Sobel,
supresión de no-máximos e histéresis por doble umbral — para producir bordes **finos,
binarios y conectados**. A diferencia de Sobel y Prewitt, el resultado no es un mapa de
gradiente continuo, sino una máscara donde cada píxel es borde o no.

El parámetro central es el par de umbrales $(T_{low}, T_{high})$. Los píxeles con gradiente por encima de
$T_{high}$ son bordes seguros; por debajo de $T_{low}$, se descartan. Los píxeles ambiguos —
entre los dos umbrales — se deciden mediante **histéresis**: se convierten en borde si están
conectados a un borde seguro, y se descartan en caso contrario. Esto evita tanto la pérdida de
tramos débiles de bordes reales como la inclusión de ruido aislado. Una heurística común
es $T_{high} = 3 \times T_{low}$.

In [ ]:
%%writefile tmp/fig_03_canny.cpp
#define MM_OUT "tmp/fig_03_canny.png"
// Compile with: g++ -std=c++17 -o output code.cpp -I<include_path>
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-canny
    //| fig-cap: "Detector de Canny con diferentes pares de umbral: umbrales bajos capturan más bordes (incluso ruido); umbrales altos retienen solo los bordes más fuertes."
    //| echo: true
    //| output: true

    mm::show(
        std::vector<mm::Image>{img_gray_crop,
             mm::canny(img_gray_crop, 30,  90),
             mm::canny(img_gray_crop, 60,  180),
             mm::canny(img_gray_crop, 120, 240)},
        MM_OUT,
        std::vector<std::string>{"Original", "Canny (30/90)", "Canny (60/180)", "Canny (120/240)"},
        4
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_canny.cpp -o tmp/fig_03_canny \
  && ./tmp/fig_03_canny \
  && test -f "tmp/fig_03_canny.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_canny.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_canny.png'
if _os.path.exists(_sent):
    mm.show(mm.read("tmp/fig_03_canny.png"))
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.22:** Detector de Canny com diferentes pares de limiar: limiares baixos capturam mais bordas (inclusive ruído); limiares altos retêm apenas as bordas mais fortes.


> ### 📝 Elección de los umbrales
>
> Una heurística común es $T_{high} = 3 \times T_{low}$. Los valores típicos dependen del rango de gradiente de la imagen — `cv2.Canny` acepta valores absolutos en $[0, 255]$. Para imágenes con contraste variable, calcular los umbrales a partir de percentiles de la magnitud de Sobel es más robusto que usar valores fijos.

## 3.7 Filtros de Orden: Filtro de la Mediana

Los filtros de orden (*order-statistic filters*) reemplazan el píxel central por el valor de un **percentil** de la distribución de intensidades de la vecindad — a diferencia de los filtros lineales, que calculan combinaciones ponderadas. El más importante es el **filtro de la mediana**.

### 3.7.1 Ruido Impulsivo: Sal y Pimienta

El ruido **sal y pimienta** (*salt-and-pepper noise*) reemplaza píxeles aleatorios por valores extremos: 0 (pimienta, negro) o 255 (sal, blanco). Es común en la transmisión de imágenes con errores de bit y en cámaras con sensores defectuosos.

Para entender por qué fallan los filtros lineales, considere una vecindad 3×3 donde un único píxel ha sido corrompido a 255:

$$
\text{vecindad} = \begin{bmatrix} 102 & 98 & 105 \\ 100 & \mathbf{255} & 97 \\ 103 & 99 & 101 \end{bmatrix}
$$

<a id="tbl-03-mediana"></a>

**Tabela 3.4:** Media vs. mediana con un píxel corrompido. La mediana ignora el valor atípico; la media se desplaza ~40 niveles.

| Método | Cálculo | Resultado |
|:-------|:--------|----------:|
| Media | (102+98+...+255+...+101)/9 | ≈ 140 |
| Mediana | {97,98,99,100,**101**,102,103,105,255} | 101 |


> ### ⚠️ ¿Por qué fallan los filtros de media con ruido impulsivo?
>
> La media es sensible a los ***valores atípicos*** — un único píxel con valor 255 en una vecindad de valor ≈ 100 eleva la salida a ≈ 140, propagando el ruido por la imagen. La mediana, por ser un **estimador robusto**, selecciona el valor central de la distribución ordenada, descartando naturalmente los extremos sin ningún ajuste especial.

El siguiente ejemplo ilustra el comportamiento de la media y de la mediana en presencia de un píxel corrupto por ruido impulsivo. Se observa que la media está fuertemente influenciada por el valor extremo (255), produciendo una estimación distante de los valores predominantes de la vecindad. En cambio, la mediana permanece cercana al valor original de la región, evidenciando su mayor robustez frente a *valores atípicos* y justificando su uso en la eliminación de ruido de sal y pimienta.

La [Figura 3.23](#fig-03-ruido) presenta el efecto del ruido sal y pimienta en diferentes densidades. El ruido fue generado reemplazando aleatoriamente una fracción de los píxeles por valores mínimos (0, pimienta) y máximos (255, sal). A medida que la densidad aumenta del 2% al 10%, crece la cantidad de píxeles corruptos, haciendo que la degradación visual sea más evidente y dificultando la percepción de los detalles de la imagen.

In [ ]:
%%writefile tmp/fig_03_ruido.cpp
#define MM_OUT "tmp/fig_03_ruido.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <random>
#include <algorithm>
#include <filesystem>

//| label: fig-03-ruido
//| fig-cap: "Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%): metade dos pixels corrompidos vira sal (255), metade pimenta (0)."
//| echo: true
//| output: true

mm::Image salt_pepper(const mm::Image& img, double prob) {
    mm::Image out = img;
    int h = out.h;
    int w = out.w;
    std::mt19937 rng(42);
    std::uniform_int_distribution<int> dist_y(0, h - 1);
    std::uniform_int_distribution<int> dist_x(0, w - 1);
    std::uniform_real_distribution<double> dist_rand(0.0, 1.0);
    int n = static_cast<int>(prob * h * w);
    for (int i = 0; i < n; i++) {
        int y = dist_y(rng);
        int x = dist_x(rng);
        if (dist_rand(rng) < 0.5)
            out.at(y, x) = 0;
        else
            out.at(y, x) = 255;
    }
    return out;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::Image n2 = salt_pepper(img_gray_crop, 0.02);
    mm::Image n5 = salt_pepper(img_gray_crop, 0.05);
    mm::Image n10 = salt_pepper(img_gray_crop, 0.10);

    mm::show(std::vector<mm::Image>{img_gray_crop, n2, n5, n10},
             MM_OUT,
             std::vector<std::string>{"Original", "Ruido 2%", "Ruido 5%", "Ruido 10%"},
             4);
    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_ruido_0.png");
mm::write(n2, "tmp/fig_03_ruido_1.png");
mm::write(n5, "tmp/fig_03_ruido_2.png");
mm::write(n10, "tmp/fig_03_ruido_3.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_ruido.cpp -o tmp/fig_03_ruido \
  && ./tmp/fig_03_ruido \
  && test -f "tmp/fig_03_ruido.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_ruido.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_ruido_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_ruido_0.png"),
            mm.read("tmp/fig_03_ruido_1.png"),
            mm.read("tmp/fig_03_ruido_2.png"),
            mm.read("tmp/fig_03_ruido_3.png"),
        ],
        titles=[
            'Original',
            'Ruido 2%',
            'Ruido 5%',
            'Ruido 10%',
        ],
        cols=4,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.23:** Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%): metade dos pixels corrompidos vira sal (255), metade pimenta (0).


### 3.7.2 Filtro de la Mediana

El filtro de la mediana sustituye cada píxel por el **valor mediano** de los píxeles de su vecindad $n \times n$:

<a id="eq-03-mediana"></a>
$$
g(x,y) = \text{med}_{(s,t) \in \mathcal{V}_{n}} \{f(x+s, y+t)\} \tag{3.24}
$$


El valor mediano es aquel que ocupa la posición central cuando los $n^2$ valores de la vecindad se ordenan. Para una ventana $3\times3$ ($n^2=9$ píxeles), la mediana es el 5º valor de la secuencia ordenada.

Para ilustrar, considere el mismo *patch* 5×5 con un píxel corrompido artificialmente en $[1,1]$:

El ejemplo confirma: incluso con el píxel corrompido a 255, la mediana devuelve el valor central correcto — el *outlier* ocupa la última posición en la ordenación y se descarta de forma natural.

Por basarse en la ordenación y no en la suma, la mediana posee tres propiedades fundamentales que la diferencian de los filtros lineales:

- **Robusta** ante el ruido impulsivo — los *outliers* van a los extremos de la secuencia ordenada y no afectan el valor central;
- **Preservadora de bordes** — las transiciones abruptas de intensidad se mantienen, pues la mediana selecciona un valor que ya existe en la vecindad, sin crear nuevos niveles intermedios;
- **No lineal** — no puede expresarse como convolución, por lo que `mm::conv` no se aplica; se utiliza `cv2.medianBlur`.

A [Figura 3.24](#fig-03-ruido-filtros) compara diferentes técnicas de remoção de ruido sal y pimienta aplicadas a una imagen con un 10% de píxeles corruptos. Se evaluaron los filtros Gaussiano, Media, Mediana, Bilateral y Morfológico (próximo capítulo), lo que permite observar la relación entre la eliminación de ruido y la preservación de detalles. En general, los filtros de media y Gaussiano reducen el ruido, pero tienden a desenfocar los bordes, mientras que la mediana presenta un mejor rendimiento para ruido impulsivo. El filtro bilateral preserva mejor los bordes, y el filtro morfológico elimina gran parte de los píxeles corruptos sin degradar excesivamente la estructura de la imagen.

In [ ]:
%%writefile tmp/fig_03_ruido_filtros.cpp
#define MM_OUT "tmp/fig_03_ruido_filtros.png"
#include "morph.hpp"
#include <random>
#include <vector>
#include <string>
#include <filesystem>

//| label: fig-03-ruido-filtros
//| fig-cap: "Filtros para ruído sal e pimenta (10%): Gaussiano, Média e Mediana. Bilateral e morfológico (open+close) ficam só na trilha Python — bilateral não tem equivalente em morph.hpp e morfologia é do próximo capítulo."
//| echo: true
//| output: true

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // Função para adicionar ruído sal e pimenta
    auto salt_pepper = [](const mm::Image& img, double prob) {
        mm::Image out = img;
        int h = out.h, w = out.w;
        std::mt19937 rng(42);
        std::uniform_int_distribution<int> dist_y(0, h - 1);
        std::uniform_int_distribution<int> dist_x(0, w - 1);
        std::uniform_real_distribution<double> dist_p(0.0, 1.0);

        for (int i = 0; i < static_cast<int>(prob * h * w); ++i) {
            int y = dist_y(rng);
            int x = dist_x(rng);
            out.at(y, x) = (dist_p(rng) < 0.5) ? 0 : 255;
        }
        return out;
    };

    mm::Image noisy = salt_pepper(img_gray_crop, 0.10);

    mm::Image f_gauss   = mm::gaussian(noisy, 5, 1.0);
    mm::Image f_media   = mm::blur(noisy, 5);
    mm::Image f_median3 = mm::median(noisy, 3);
    mm::Image f_median5 = mm::median(noisy, 5);

    mm::show(
        std::vector<mm::Image>{img_gray_crop, noisy, f_gauss, f_media, f_median3, f_median5},
        MM_OUT,
        std::vector<std::string>{"Original", "Ruido 10%", "Gaussiano 5x5", "Media 5x5",
                                 "Mediana 3x3", "Mediana 5x5"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_ruido_filtros_0.png");
mm::write(noisy, "tmp/fig_03_ruido_filtros_1.png");
mm::write(f_gauss, "tmp/fig_03_ruido_filtros_2.png");
mm::write(f_media, "tmp/fig_03_ruido_filtros_3.png");
mm::write(f_median3, "tmp/fig_03_ruido_filtros_4.png");
mm::write(f_median5, "tmp/fig_03_ruido_filtros_5.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_ruido_filtros.cpp -o tmp/fig_03_ruido_filtros \
  && ./tmp/fig_03_ruido_filtros \
  && test -f "tmp/fig_03_ruido_filtros.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_ruido_filtros.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_ruido_filtros_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_ruido_filtros_0.png"),
            mm.read("tmp/fig_03_ruido_filtros_1.png"),
            mm.read("tmp/fig_03_ruido_filtros_2.png"),
            mm.read("tmp/fig_03_ruido_filtros_3.png"),
            mm.read("tmp/fig_03_ruido_filtros_4.png"),
            mm.read("tmp/fig_03_ruido_filtros_5.png"),
        ],
        titles=[
            'Original',
            'Ruido 10%',
            'Gaussiano 5x5',
            'Media 5x5',
            'Mediana 3x3',
            'Mediana 5x5',
        ],
        cols=3,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.24:** Filtros para ruído sal e pimenta (10%): Gaussiano, Média e Mediana. Bilateral e morfológico (open+close) ficam só na trilha Python — bilateral não tem equivalente em morph.hpp e morfologia é do próximo capítulo.


## 3.8 Aplicación Práctica: Preprocesamiento para Segmentación

En la práctica, las técnicas de este capítulo rara vez se utilizan de forma aislada. Un ***pipeline* de preprocesamiento** típico combina varias etapas en secuencia, adaptándose al tipo de imagen y a la aplicación. La [Figura 3.25](#fig-03-pipeline) ilustra un *pipeline* completo:

1. **Ecualización de histograma (CLAHE):** normaliza el contraste independientemente de las condiciones de iluminación;
2. **Filtro Gaussiano:** suaviza el ruido de adquisición sin destruir bordes;
3. **Detección de bordes (Sobel/Canny):** extrae estructuras relevantes para la segmentación.

> ### 📝 El orden importa
>
> El orden de las operaciones afecta el resultado final. En general: **(1) normalización de intensidad → (2) reducción de ruido → (3) realce/segmentación**. Invertir el orden puede amplificar el ruido o perder bordes antes de detectarlos.

In [ ]:
%%writefile tmp/fig_03_pipeline.cpp
#define MM_OUT "tmp/fig_03_pipeline.png"
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-pipeline
    //| fig-cap: "*Pipeline* de pré-processamento: equalização → Gaussiano → Canny. A trilha Python usa CLAHE no lugar da equalização global; CLAHE não tem equivalente em morph.hpp."
    //| echo: true
    //| output: true

    mm::Image img_eq    = mm::equalize(img_gray_crop);     // Etapa 1: equalização global
    mm::Image img_gauss = mm::gaussian(img_eq, 5, 0);      // Etapa 2: Gaussiano
    mm::Image edges     = mm::canny(img_gauss, 50, 150);   // Etapa 3: Canny

    mm::Image edges_direct = mm::canny(img_gray_crop, 50, 150);   // Canny direto, sem pré-processo

    mm::show(
        std::vector<mm::Image>{img_gray_crop, img_eq, img_gauss, edges, edges_direct},
        MM_OUT,
        std::vector<std::string>{"Original", "1. Equalizado", "2. Gaussiano", "3. Canny (pipeline)", "Canny (direto)"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_pipeline_0.png");
mm::write(img_eq, "tmp/fig_03_pipeline_1.png");
mm::write(img_gauss, "tmp/fig_03_pipeline_2.png");
mm::write(edges, "tmp/fig_03_pipeline_3.png");
mm::write(edges_direct, "tmp/fig_03_pipeline_4.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 tmp/fig_03_pipeline.cpp -o tmp/fig_03_pipeline \
  && ./tmp/fig_03_pipeline \
  && test -f "tmp/fig_03_pipeline.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_pipeline.png"

In [ ]:
import os as _os
_sent = 'tmp/fig_03_pipeline_0.png'
if _os.path.exists(_sent):
    mm.show(
        [
            mm.read("tmp/fig_03_pipeline_0.png"),
            mm.read("tmp/fig_03_pipeline_1.png"),
            mm.read("tmp/fig_03_pipeline_2.png"),
            mm.read("tmp/fig_03_pipeline_3.png"),
            mm.read("tmp/fig_03_pipeline_4.png"),
        ],
        titles=[
            'Original',
            '1. Equalizado',
            '2. Gaussiano',
            '3. Canny (pipeline)',
            'Canny (direto)',
        ],
        cols=5,
    )
else:
    print("figura indisponivel nesta trilha: o programa C++ compilou mas nao gerou " + _sent + " (ver a versao Python)")

**Figura 3.25:** *Pipeline* de pré-processamento: equalização → Gaussiano → Canny. A trilha Python usa CLAHE no lugar da equalização global; CLAHE não tem equivalente em morph.hpp.


## 3.9 Resumen

En este capítulo se presentaron las principales técnicas de procesamiento en el dominio espacial, desde la manipulación directa de píxeles hasta el filtrado por vecindad:

- **Operaciones de punto:** aritméticas saturadas (`mm::addm`, `mm::subm`) y lógicas bit a bit (`mm::band`, `mm::bor`, `mm::bnot`) para recorte de ROI y combinación de imágenes; *alpha blending* (`mm::blend`) para fusión ponderada con peso $\alpha \in [0,1]$.
- **Histograma:** función discreta de distribución de intensidades; visualizado con `mm::histImg` y calculado con `mm::hist`; base para el diagnóstico tonal y para las técnicas de ecualización y especificación.
- **Ecualización:** redistribución automática de las intensidades mediante la CDF (`mm::equalize`), con variante adaptativa CLAHE para el control local del contraste.
- **Especificación de histograma:** transferencia del perfil tonal de una imagen de referencia mediante mapeo inverso de la CDF — generalización de la ecualización para distribuciones arbitrarias.
- **Correlación y convolución:** mecanismo de ventana deslizante implementado en `mm::conv` (`cv2.filter2D`); diferenciados por la rotación de 180° del *kernel* — relevante solo para kernels asimétricos.
- **Filtros de suavizado:** media (*kernel* uniforme, desenfoca bordes proporcionalmente al tamaño) y Gaussiano (ponderación radial, separable, sin *ringing*, conserva mejor los bordes).
- **Filtros de realce:** Laplaciano ($w_4$/$w_8$, segunda derivada isotrópica), Sobel (gradiente direccional de primer orden, con magnitud $|\nabla f|$ y dirección $\theta$) y *Unsharp Masking* (amplificación de las altas frecuencias con parámetro $k$).
- **Filtro de la mediana:** no lineal, robusto a valores atípicos, conserva bordes — superior a los filtros lineales para ruido sal y pimienta.
- **Pipeline práctico:** encadenamiento CLAHE → Gaussiano → Canny como estrategia de preprocesamiento; `mm::drawImgKernel` para visualización didáctica de la ventana deslizante.

El Capítulo 4 abordará la **morfología matemática** (erosión, dilatación, apertura y cierre), explorando en profundidad las funciones `mm::ero` y `mm::dil` de la biblioteca `morph.py`. A continuación, el Capítulo 5 presentará el **procesamiento en el dominio de la frecuencia**, con enfoque en la Transformada de Fourier y en técnicas de filtrado espectral.

## 3.10 🤖 Uso del Gemini Notebook como Tutor Complementario

En esta edición, incentivamos el uso del **Gemini Notebook** como herramienta complementaria de aprendizaje. Esta herramienta de IA utiliza exclusivamente los documentos proporcionados por el autor como base de conocimiento, garantizando respuestas coherentes con el contenido del libro — incluyendo las funciones de la biblioteca `morph.py` y los experimentos realizados en este capítulo.

Para cada capítulo, hemos preparado un proyecto específico en la plataforma con el PDF del capítulo, los *notebooks* y materiales auxiliares. Sugerimos explorar especialmente:

- **Guía de Estudio:** resumen estructurado de los conceptos, ideal para repasar antes de los exámenes;
- **Conversación:** resuelve dudas sobre ecualización, convolución, filtros y pipelines directamente con el tutor;
- **Preguntas frecuentes:** cuestiones típicas sobre la diferencia entre media y mediana, USM, Lapaciano vs. Sobel.

> ### ❗ 🎓 Estudia con el Tutor Inteligente
>
> Para interactuar con el contenido de este capítulo, accede al siguiente *enlace*. El entorno contiene materiales didácticos en diferentes formatos, generados a partir del **PDF** del capítulo. En la plataforma, explora especialmente las opciones **Guía de Estudio** y **Conversación** para profundizar tu comprensión.
>
> [🚀 ACCEDER A GEMINI NOTEBOOK: CAPÍTULO 03](https://notebooklm.google.com/notebook/d6593e26-a008-4d3b-8073-5c9b7d00eacc)
>
> #### 🌐 Idioma y Lenguaje de Programación
>
> El proyecto de este capítulo en Gemini Notebook fue construido únicamente con el texto en **portugués** y los ejemplos de código en **Python**. Si estás estudiando con la edición en inglés o francés, o siguiendo la ruta en C++, las respuestas del tutor pueden no corresponder exactamente con la versión que estás leyendo.
>
> #### ⚠️ Aviso sobre Contenido Generado por IA
>
> La IA es una poderosa aliada en los estudios, pero el contenido generado puede contener **errores o imprecisiones**. Consulta siempre **libros, artículos científicos y otras fuentes académicas confiables** para validar la información. Siempre que sea posible, ejecuta los ejemplos prácticos proporcionados en este capítulo para verificar los resultados.

## 3.11 Lista de Ejercicios

1. **(10%)** Explique la diferencia entre **convolución** y **correlación cruzada**. ¿Para qué tipos de *kernel* los resultados son idénticos? Dé un ejemplo de *kernel* asimétrico (como Sobel $G_x$) y muestre numéricamente que los resultados difieren aplicándolo al parche 5×5 del capítulo de las dos formas.

2. **(15%)** Considere una imagen 5×5 con intensidades concentradas entre los niveles 3 y 5 (bajo contraste, 3 bits). Aplique manualmente el algoritmo de ecualización de la [Tabela 3.1](#tbl-03-equalizacao), completando todas las columnas de la tabla ($k$, $h[k]$, $p[k]$, $\text{cdf}[k]$, $\text{lut}[k]$). Verifique el resultado con `mm::equalize`.

3. **(15%)** Usando `mm::conv`, aplique el filtro de media con kernels de tamaño 3×3, 9×9 y 21×21 a la imagen del mandril. Para cada versión, calcule el **PSNR** (*Peak Signal-to-Noise Ratio*) con respecto a la original:
$$\text{PSNR} = 10\log_{10}\!\left(\frac{255^2}{\text{MSE}}\right), \quad \text{MSE} = \frac{1}{MN}\sum_{i,j}(f-g)^2$$
Grafique el PSNR en función del tamaño del kernel y explique qué indica la caída progresiva sobre la relación entre suavizado y pérdida de información.

4. **(15%)** Usando `add_salt_pepper` con densidad del 5%, aplique y compare: (a) `mm::conv` con media 3×3, (b) `cv2.GaussianBlur` con $\sigma=1$, (c) `cv2.medianBlur` con ventana 3×3 y (d) `cv2.medianBlur` con ventana 5×5. Muestre las imágenes con `mm::show` en una cuadrícula 2×4 (fila 1: imágenes, fila 2: histogramas mediante `mm::histImg`). Explique por qué la mediana supera a los filtros lineales usando el argumento de la [Tabela 3.4](#tbl-03-mediana).

5. **(15%)** Implemente `mm::conv0` usando solo operaciones NumPy vectorizadas — sin bucles en Python y sin `cv2.filter2D` — con el operador de *stride tricks* (`np.lib.stride_tricks.sliding_window_view`). Compare el resultado y el tiempo de ejecución con `mm::conv0` (bucles) y `mm::conv` (cv2) para kernels 3×3 y 15×15 en la imagen del mandril.

6. **(15%)** Aplique el *Unsharp Masking* con $\sigma=1$ y $k \in \{0.5, 1.0, 2.0, 4.0\}$ usando la función `usm` del capítulo. Para cada valor de $k$: (a) calcule la diferencia absoluta $|g - f|$, (b) muestre las imágenes y las diferencias con `mm::show`, y (c) grafique el histograma de las diferencias con `mm::histImg`. Identifique a partir de cuál $k$ los artefactos (halos y amplificación de ruido) se vuelven visualmente inaceptables.

7. **(15%)** Elija una imagen de rayos X o tomografía disponible públicamente (ej.: mediante `mm::read` de URL) y diseñe un pipeline de preprocesamiento con al menos 4 etapas secuenciales, justificando cada elección con base en los conceptos del capítulo. Muestre con `mm::show` en una cuadrícula: imagen original, cada etapa intermedia y el resultado final con sus histogramas (`mm::histImg`).

## Referencias del Capítulo

La fundamentación teórica de este capítulo se basa en las siguientes obras:

* Gonzalez (2018) para los conceptos de operaciones de intensidad, histograma, convolución y filtrado espacial.
* Szeliski (2022) para la visión por computadora y aplicaciones prácticas de filtrado.
* Bradski (2008) para la implementación práctica con OpenCV y `morph.py`.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.es/cap03/cap03.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 3.12 💻 **Parte Práctica con Ejercicios de Programación**

### 🎯 Objetivo de este Cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

#### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda a continuación:

In [ ]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build del trayecto C++ (.cpp, binario, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# El kernel es Python incluso en el trayecto C++: `mm` (morph.py) es usado por los
# simuladores, por la exhibición de las figuras que el binario C++ genera y por el
# estado mm::Image entre celdas. cpp=True descarga también el trayecto compilado
# (morph.hpp + stb_image*.h), usado en el #include de las celdas %%writefile *.cpp.
import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando los Tests
Para evaluar los tests, ejecuta `TestSuite("EP03_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar un archivo, usa `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
from morph import mm
# 3 ... tu código aquí ...
"""
TestSuite("EP03_01").run_code(codigo)
```

### 3.0.1 EP03_01 ➕ Adición Saturada de Constante

En sistemas de vigilancia por video, las cámaras en entornos con iluminación variable producen imágenes subexpuestas. El ajuste de brillo mediante **adición saturada de una constante** es la operación más simple para la corrección inmediata, aplicándose en tiempo real en los *chips* de cámaras embebidas y en *pipelines* de preprocesamiento de robots móviles.

Ver en [Figura 3.26](#fig-03-sim-ep0301-adicao) una simulación de este EP.


#### 3.0.1.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Constante:** Leer el entero $k$ (valor a sumar).
3. **Datos:** Leer los valores enteros de la matriz original fila por fila.
4. **Mapeo:** Para cada píxel $p$, calcular el nuevo valor mediante la ecuación:

$$p' = \text{clip}(p + k)$$

5. **Salida:** Mostrar la matriz resultante con dimensiones $L \times C$.

#### 3.0.1.2 📌 Restricciones Computacionales

* **Saturación (*Clipping*):** Los valores deben confinarse al intervalo $[0, 255]$:
$$\text{clip}(x) = \max(0, \min(255, x))$$
* **Tipo:** El resultado final debe ser entero (sin decimales).
* **$k$ puede ser negativo:** los valores negativos oscurecen la imagen; los positivos la aclaran.

#### 3.0.1.3 🧠 Fundamentación Teórica

| Parámetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$k > 0$** | Entero | Aclara la imagen; los píxeles cercanos a 255 saturan en blanco |
| **$k < 0$** | Entero | Oscurece la imagen; los píxeles cercanos a 0 saturan en negro |
| **$k = 0$** | Entero | Imagen sin cambios |

#### 3.0.1.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $k$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz transformada en $L$ filas y $C$ columnas, valores enteros separados por espacios.

#### 3.0.1.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 2<br>3<br>50<br>0 100 200<br>210 240 255 | 50 150 250<br>255 255 255 | Saturación en 255 en los píxeles altos |
| 1<br>4<br>-30<br>0 20 200 255 | 0 0 170 225 | Saturación en 0 en los píxeles bajos |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0301-adicao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">➕ Simulador EP03_01: Adición Saturada de Constante</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = clip(p + k)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajuste el valor de la constante k para observar el desplazamiento de brillo de la imagen y el truncamiento por saturación en el intervalo [0, 255].</p>

    <!-- Controle da Constante k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#27ae60;">Constante (k)</label>
        <span id="sim_ep0301_vl_k" style="font-family:monospace;font-size:12px;font-weight:700;color:#27ae60;">0</span>
      </div>
      <input type="range" id="sim_ep0301_sl_k" min="-128" max="128" step="1" value="0" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (p)</span>
        <div id="sim_ep0301_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Resultado Transformado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Transformado (p')</span>
        <div id="sim_ep0301_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Restablecer (k = 0)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0301_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>clip(p + (0))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0301(root){
    if (!root || root.dataset.simEp0301Init) return;
    root.dataset.simEp0301Init = "1";

    var slK      = root.querySelector('#sim_ep0301_sl_k');
    var vlK      = root.querySelector('#sim_ep0301_vl_k');
    var gridOrig = root.querySelector('#sim_ep0301_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0301_grid_new');
    var debugDiv = root.querySelector('#sim_ep0301_debug');

    var btnNew   = root.querySelector('#sim_ep0301_btnNew');
    var btnReset = root.querySelector('#sim_ep0301_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function render() {
      var k = parseInt(slK.value) || 0;
      vlK.textContent = k;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>clip(p + (' + k + '))</b>';

      gridOrig.innerHTML = '';
      gridNew.innerHTML  = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        // Célula Resultado
        var res = Math.max(0, Math.min(255, p + k));
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slK.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slK.value = 0;
      render();
    });

    render();
  }

  function tryInitSimEP0301(){
    var root = document.getElementById('sim-ep0301-adicao');
    if (root) initSimEP0301(root); else setTimeout(tryInitSimEP0301, 200);
  }
  tryInitSimEP0301();
})();
</script>
</div>
""")

**Figura 3.26:** Simulador EP03_01: Adición Saturada de Constante (p


<figure id="fig-03-sim-ep0301-adicao">
  <img src="imagens/fig-03-sim-ep0301-adicao.png" alt=" Simulador EP03_01: Adición Saturada de Constante (p' = clip(p + k)) " style="max-width:80%" />
  <figcaption><strong>Figura 3.26:</strong>  Simulador EP03_01: Adición Saturada de Constante (p' = clip(p + k)) </figcaption>
</figure>

In [ ]:
%%writefile EP03_01.cpp
// your solution

In [ ]:
TestSuite("EP03_01.cpp").run()

### 3.0.2 EP03_02 🔀 Alpha *Blending* de Dos Imágenes

En medicina nuclear, imágenes de diferentes modalidades (tomografía computarizada y resonancia magnética) se fusionan para ayudar en el diagnóstico. La **mezcla ponderada** (*alpha blending*) es la operación fundamental de este proceso, permitiendo al radiólogo controlar interactivamente el peso de cada modalidad en la imagen mostrada.

Ver en [Figura 3.27](#fig-03-sim-ep0302-blending) una simulación de este EP.


#### 3.0.2.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Parámetro:** Leer el valor real $\alpha \in [0, 1]$.
3. **Datos:** Leer los valores enteros de la matriz $f_1$ (imagen 1) y luego de la matriz $f_2$ (imagen 2).
4. **Mapeo:** Para cada posición $(i, j)$, calcular:

$$g(i,j) = \text{clip}\left(\text{round}\left(\alpha \cdot f_1(i,j) + (1-\alpha) \cdot f_2(i,j)\right)\right)$$

5. **Salida:** Mostrar la matriz resultante $L \times C$.

#### 3.0.2.2 📌 Restricciones Computacionales

* **Redondeo:** Aplicar `round` antes de la conversión a entero.
* **Saturación:** Confinar al intervalo $[0, 255]$ con $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operación en float:** Realizar la operación en punto flotante antes de redondear.

#### 3.0.2.3 🧠 Fundamentación Teórica

| Valor de $\alpha$ | Resultado |
|:-----------------:|:----------|
| $\alpha = 1.0$ | Solo $f_1$ |
| $\alpha = 0.5$ | Media aritmética de $f_1$ y $f_2$ |
| $\alpha = 0.0$ | Solo $f_2$ |

#### 3.0.2.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Real $\alpha$.
* Líneas siguientes: Elementos de $f_1$ ($L$ líneas con $C$ valores cada una).
* Líneas siguientes: Elementos de $f_2$ ($L$ líneas con $C$ valores cada una).

**Salida:**

* Matriz resultante $L \times C$.

#### 3.0.2.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 1<br>3<br>0.5<br>0 100 200<br>100 200 50 | 50 150 125 | Media entre las dos imágenes |
| 1<br>3<br>1.0<br>10 20 30<br>90 80 70 | 10 20 30 | Solo $f_1$ (alpha=1) |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0302-blending" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔀 Simulador EP03_02: Alpha Blending de Dos Imágenes</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = α·f1 + (1−α)·f2</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajusta el parámetro de transparencia α para observar la combinación lineal ponderada píxel a píxel entre las imágenes f1 y f2.</p>

    <!-- Controle do Parâmetro Alpha -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">α (Alpha — Peso de f1)</label>
        <span id="sim_ep0302_vl_a" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;">0.50</span>
      </div>
      <input type="range" id="sim_ep0302_sl_a" min="0" max="1" step="0.05" value="0.5" style="width:100%;cursor:pointer;">
      <div style="margin-top:6px;font-size:10px;color:#8a8371;text-align:center;font-family:monospace;">
        α = 0.00 → Solo f2 &nbsp;|&nbsp; α = 0.50 → Media Ponderada Igual &nbsp;|&nbsp; α = 1.00 → Solo f1
      </div>
    </div>

    <!-- Comparativo em 3 Colunas: f1 vs f2 vs Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(160px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f1 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen f1</span>
        <div id="sim_ep0302_grid_f1" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuevas Imágenes</button>
      </div>

      <!-- Imagem f2 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen f2</span>
        <div id="sim_ep0302_grid_f2" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado g</span>
        <div id="sim_ep0302_grid_g" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Restablecer (α = 0.5)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0302_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula: <b>clip(round(0.50 · f1 + 0.50 · f2))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0302(root){
    if (!root || root.dataset.simEp0302Init) return;
    root.dataset.simEp0302Init = "1";

    var slA      = root.querySelector('#sim_ep0302_sl_a');
    var vlA      = root.querySelector('#sim_ep0302_vl_a');
    var gF1      = root.querySelector('#sim_ep0302_grid_f1');
    var gF2      = root.querySelector('#sim_ep0302_grid_f2');
    var gG       = root.querySelector('#sim_ep0302_grid_g');
    var debugDiv = root.querySelector('#sim_ep0302_debug');

    var btnNew   = root.querySelector('#sim_ep0302_btnNew');
    var btnReset = root.querySelector('#sim_ep0302_btnReset');

    var px1 = [], px2 = [];

    function generate() {
      px1 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
      px2 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function createCell(val) {
      var c = document.createElement('div');
      var fgColor = val > 128 ? '#000000' : '#ffffff';
      c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + val + ',' + val + ',' + val + ');color:' + fgColor + ';box-sizing:border-box;';
      c.textContent = val;
      return c;
    }

    function render() {
      var a = parseFloat(slA.value) || 0;
      var a1 = a.toFixed(2);
      var a2 = (1 - a).toFixed(2);

      vlA.textContent = a1;
      debugDiv.innerHTML = 'Fórmula: <b>clip(round(' + a1 + ' · f1 + ' + a2 + ' · f2))</b>';

      gF1.innerHTML = '';
      gF2.innerHTML = '';
      gG.innerHTML  = '';

      for (var i = 0; i < 16; i++) {
        gF1.appendChild(createCell(px1[i]));
        gF2.appendChild(createCell(px2[i]));

        var res = Math.max(0, Math.min(255, Math.round(a * px1[i] + (1 - a) * px2[i])));
        gG.appendChild(createCell(res));
      }
    }

    slA.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    btnReset.addEventListener('click', function() {
      slA.value = '0.5';
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0302(){
    var root = document.getElementById('sim-ep0302-blending');
    if (root) initSimEP0302(root); else setTimeout(tryInitSimEP0302, 200);
  }
  tryInitSimEP0302();
})();
</script>
</div>
""")

**Figura 3.27:** Simulador EP03_02: Mezcla alfa de dos imágenes (g = α·f1 + (1−α)·f2)


<figure id="fig-03-sim-ep0302-blending">
  <img src="imagens/fig-03-sim-ep0302-blending.png" alt=" Simulador EP03_02: Mezcla alfa de dos imágenes (g = α·f1 + (1−α)·f2) " style="max-width:80%" />
  <figcaption><strong>Figura 3.27:</strong>  Simulador EP03_02: Mezcla alfa de dos imágenes (g = α·f1 + (1−α)·f2) </figcaption>
</figure>

In [ ]:
%%writefile EP03_02.cpp
// your solution

In [ ]:
TestSuite("EP03_02.cpp").run()

### 3.0.3 EP03_03 🎭 Inversión de Imagen (Negativo Fotográfico)

En radiología, las imágenes de rayos X se visualizan tradicionalmente en negativo: los huesos aparecen en negro sobre fondo blanco. La operación de **negativo fotográfico** se aplica rutinariamente en PACS (*Picture Archiving and Communication Systems*) para facilitar la detección de fracturas y densidades óseas.

Ver en [Figura 3.28](#fig-03-sim-ep0303-inversao) una simulación de este EP.


#### 3.0.3.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer los valores enteros de la matriz original.
3. **Mapeo:** Para cada píxel $p$, calcular el negativo:

$$p' = 255 - p$$

4. **Salida:** Mostrar la matriz resultante de $L \times C$.

#### 3.0.3.2 📌 Restricciones Computacionales

* **Sin necesidad de *recorte*:** El resultado de $255 - p$ con $p \in [0, 255]$ siempre está $\in [0, 255]$.
* **Tipo entero:** La salida debe ser valores enteros.
* **Equivalencia lógica:** La operación es idéntica al `NOT` bit a bit (`mm::bnot`) en imágenes de 8 bits.

#### 3.0.3.3 🧠 Fundamentación Teórica

| Píxel Original $p$ | Píxel Negativo $p'$ | Observación |
|:------------------:|:-------------------:|:----------:|
| 0 (negro) | 255 (blanco) | Inversión total |
| 128 (gris medio) | 127 (gris medio) | Valor central |
| 255 (blanco) | 0 (negro) | Inversión total |

#### 3.0.3.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos enteros de la matriz original.

**Salida:**

* Matriz negativa en $L$ filas y $C$ columnas.

#### 3.0.3.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 1<br>4<br>0 128 200 255 | 255 127 55 0 | Inversión de cada píxel |
| 2<br>2<br>10 20<br>30 40 | 245 235<br>225 215 | Matriz 2x2 invertida |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0303-inversao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎭 Simulador EP03_03: Negativo Fotográfico (Inversión)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = 255 − p</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Observe la inversión complementaria de intensidad: los tonos oscuros se vuelven claros y los tonos claros se vuelven oscuros restando cada píxel del valor máximo de 255.</p>

    <!-- Comparativo Lado a Lado: Entrada vs Negativo -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original (p)</span>
        <div id="sim_ep0303_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0303_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Negativo (p' = 255 - p) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Negativo (p' = 255 − p)</span>
        <div id="sim_ep0303_grid_neg" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel de Informação Explicativo -->
    <div id="sim_ep0303_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Fórmula aplicada: <b>p' = 255 − p</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0303(root){
    if (!root || root.dataset.simEp0303Init) return;
    root.dataset.simEp0303Init = "1";

    var gO = root.querySelector('#sim_ep0303_grid_orig');
    var gN = root.querySelector('#sim_ep0303_grid_neg');
    var btnNew = root.querySelector('#sim_ep0303_btnNew');

    var pixels = [];

    function generate() {
      pixels = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function render() {
      gO.innerHTML = '';
      gN.innerHTML = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gO.appendChild(cellO);

        // Célula Negativo
        var r = 255 - p;
        var fgColorN = r > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = r;
        gN.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0303(){
    var root = document.getElementById('sim-ep0303-inversao');
    if (root) initSimEP0303(root); else setTimeout(tryInitSimEP0303, 200);
  }
  tryInitSimEP0303();
})();
</script>
""")

**Figura 3.28:** Simulador EP03_03: Inversión de Imagen — Negativo Fotográfico (p


<figure id="fig-03-sim-ep0303-inversao">
  <img src="imagens/fig-03-sim-ep0303-inversao.png" alt=" Simulador EP03_03: Inversión de Imagen — Negativo Fotográfico (p' = 255 − p) " style="max-width:80%" />
  <figcaption><strong>Figura 3.28:</strong>  Simulador EP03_03: Inversión de Imagen — Negativo Fotográfico (p' = 255 − p) </figcaption>
</figure>

In [ ]:
%%writefile EP03_03.cpp
// your solution

In [ ]:
TestSuite("EP03_03.cpp").run()

### 3.0.4 EP03_04 📊 Equalización de Histograma (L bits)

En imágenes de satélite de teleobservación, la variación de iluminación a lo largo del día produce imágenes de bajo contraste. La **equalización de histograma** se aplica automáticamente en satélites como el Landsat para redistribuir los tonos, revelando detalles de vegetación, relieve y zonas urbanas invisibles en la imagen original.

Ver en la [Figura 3.29](#fig-03-sim-ep0304-equalizacao) una simulación de este EP.


#### 3.0.4.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (líneas), $C$ (columnas) y $B$ (número de bits, con $L_{\max} = 2^B$).
2. **Datos:** Leer la matriz de píxeles $f$ con valores en $[0, 2^B - 1]$.
3. **Histograma:** Calcular $h[k]$ = número de píxeles con intensidad $k$, para $k = 0 \ldots 2^B-1$.
4. **Probabilidad:** $p[k] = h[k] / (L \cdot C)$.
5. **CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$; función de distribución acumulada.
6. **LUT:** $\text{lut}[k] = \text{round}\left(\text{cdf}[k] \cdot (2^B - 1)\right)$; *Look-Up Table* (tabla de consulta).
7. **Aplicación:** $g[i,j] = \text{lut}[f[i,j]]$.
8. **Salida:** Mostrar la matriz equalizada $L \times C$.

#### 3.0.4.2 📌 Restricciones Computacionales

* **Redondeo:** Usar redondeo matemático (`round`) en la LUT.
* **Bits:** El número de niveles es $2^B$ (ej.: $B=3 \Rightarrow 8$ niveles, $B=8 \Rightarrow 256$ niveles).
* **CDF acumulada:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$, con $\text{cdf}[2^B-1] = 1.0$.

#### 3.0.4.3 🧠 Fundamentación Teórica

| Etapa | Operación | Fórmula |
|:-----:|:---------|:--------|
| 1 | Histograma | $h[k] \leftarrow$ nº píxeles con intensidad $k$ |
| 2 | Probabilidad | $p[k] = h[k] / (L \cdot C)$ |
| 3 | CDF | $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$ |
| 4 | LUT | $\text{lut}[Look-Up Table (tabla de consulta)k] = \text{round}(\text{cdf}[k] \cdot (2^B-1))$ |
| 5 | Aplicación | $g[i,j] = \text{lut}[f[i,j]]$ |

#### 3.0.4.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $B$ (número de bits).
* Líneas siguientes: Elementos enteros de la matriz.

**Salida:**

* Matriz equalizada en $L$ líneas y $C$ columnas.

#### 3.0.4.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 5<br>5<br>3<br>3 4 2 3 4<br>4 3 3 4 3<br>2 3 4 3 2<br>3 4 3 2 3<br>4 3 2 3 4 | 5 7 1 5 7<br>7 5 5 7 5<br>1 5 7 5 1<br>5 7 5 1 5<br>7 5 1 5 7 | Ejemplo 3 bits del capítulo |
| 1<br>4<br>3<br>0 0 7 7 | 0 0 7 7 | Histograma bimodal extremo |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0304-equalizacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulador EP03_04: Ecualización de Histograma</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">lut[k] = round(cdf[k] · (L − 1))</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Elija la profundidad de bits (B) y genere imágenes para analizar la dispersión dinámica del histograma y la tabla de remapeo (LUT) en tiempo real.</p>

    <!-- Controle de Bits B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#16a085;">Profundidad de Bits (B)</label>
        <span id="sim_ep0304_vl_bits" style="font-family:monospace;font-size:12px;font-weight:700;color:#16a085;">3 bits → 8 niveles</span>
      </div>
      <input type="range" id="sim_ep0304_sl_bits" min="1" max="8" step="1" value="3" style="width:100%;cursor:pointer;">
      <div style="display:flex;justify-content:space-between;margin-top:6px;font-size:10px;color:#8a8371;font-family:monospace;">
        <span>1 bit (2 niveles)</span>
        <span>4 bits (16 niveles)</span>
        <span>8 bits (256 niveles)</span>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Equalizada -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrada Original</span>
        <div id="sim_ep0304_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Resultado Equalizado -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado Ecualizado</span>
        <div id="sim_ep0304_grid_eq" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Nuevo Muestreo</button>
      </div>

    </div>

    <!-- Histogramas Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:16px;margin-bottom:16px;">
      
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Histograma Original</span>
        <canvas id="sim_ep0304_hist_orig" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Histograma Ecualizado</span>
        <canvas id="sim_ep0304_hist_eq" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

    </div>

    <!-- Tabela LUT -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px 14px;margin-bottom:14px;overflow-x:auto;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;">LUT (Tabla de Remapeo k → v)</span>
      <div id="sim_ep0304_lut_table" style="font-family:monospace;font-size:11px;color:#26241d;white-space:nowrap;"></div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0304_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      lut[k] = round(cdf[k] · 7) | B=3, niveles=8
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0304(root){
    if (!root || root.dataset.simEp0304Init) return;
    root.dataset.simEp0304Init = "1";

    var slBits     = root.querySelector('#sim_ep0304_sl_bits');
    var vlBits     = root.querySelector('#sim_ep0304_vl_bits');
    var debugDiv   = root.querySelector('#sim_ep0304_debug');
    var gridOrig   = root.querySelector('#sim_ep0304_grid_orig');
    var gridEq     = root.querySelector('#sim_ep0304_grid_eq');
    var lutTable   = root.querySelector('#sim_ep0304_lut_table');
    var histOrig   = root.querySelector('#sim_ep0304_hist_orig');
    var histEq     = root.querySelector('#sim_ep0304_hist_eq');

    var btnNew     = root.querySelector('#sim_ep0304_btnNew');
    var btnReset   = root.querySelector('#sim_ep0304_btnReset');

    var ROWS = 4, COLS = 4;
    var pixels = [];

    function generatePixels() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      var lo = Math.floor(levels * 0.2);
      var hi = Math.floor(levels * 0.5);
      pixels = Array.from({ length: ROWS * COLS }, function(){
        return lo + Math.floor(Math.random() * (hi - lo + 1));
      });
    }

    function computeEqualization(pixArr, b) {
      var levels = Math.pow(2, b);
      var N = pixArr.length;
      var h = new Array(levels).fill(0);
      pixArr.forEach(function(p){ h[p]++; });

      var cdf = new Array(levels).fill(0);
      cdf[0] = h[0] / N;
      for (var k = 1; k < levels; k++) {
        cdf[k] = cdf[k - 1] + h[k] / N;
      }

      var lut = cdf.map(function(c){ return Math.round(c * (levels - 1)); });
      var result = pixArr.map(function(p){ return lut[p]; });
      return { h: h, cdf: cdf, lut: lut, result: result };
    }

    function drawHistogram(canvas, counts, levels, color) {
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var maxVal = Math.max.apply(null, counts.concat([1]));
      var barW = W / levels;

      counts.forEach(function(c, i){
        var barH = (c / maxVal) * (H - 6);
        ctx.fillStyle = color;
        ctx.fillRect(i * barW + 1, H - barH, barW - 2, barH);
      });
    }

    function toGray(val, levels) {
      return Math.round((val / (levels - 1)) * 255);
    }

    function render() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      vlBits.textContent = b + ' bit' + (b > 1 ? 's' : '') + ' → ' + levels + ' níveis';

      var eqData = computeEqualization(pixels, b);
      var hOrig = eqData.h;
      var lut = eqData.lut;
      var result = eqData.result;

      var hEq = new Array(levels).fill(0);
      result.forEach(function(p){ hEq[p]++; });

      lutTable.innerHTML = lut.map(function(v, k){
        return '<span style="display:inline-block;margin-right:10px;color:#8a8371;">' + k + ' → <b style="color:#16a085;">' + v + '</b></span>';
      }).join('');

      debugDiv.innerHTML = '<b>lut[k] = round(cdf[k] · ' + (levels - 1) + ')</b> &nbsp;|&nbsp; B = ' + b + ', níveis = ' + levels;

      gridOrig.innerHTML = '';
      gridEq.innerHTML = '';

      pixels.forEach(function(p, i) {
        var grayO = toGray(p, levels);
        var fgO = grayO > 128 ? '#000000' : '#ffffff';
        var cellO = document.createElement('div');
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + grayO + ',' + grayO + ',' + grayO + ');color:' + fgO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        var r = result[i];
        var grayR = toGray(r, levels);
        var fgR = grayR > 128 ? '#000000' : '#ffffff';
        var cellR = document.createElement('div');
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + grayR + ',' + grayR + ',' + grayR + ');color:' + fgR + ';box-sizing:border-box;';
        cellR.textContent = r;
        gridEq.appendChild(cellR);
      });

      drawHistogram(histOrig, hOrig, levels, '#95a5a6');
      drawHistogram(histEq, hEq, levels, '#16a085');
    }

    slBits.addEventListener('input', function(){
      generatePixels();
      render();
    });

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      generatePixels();
      render();
    });

    generatePixels();
    render();
  }

  function tryInitSimEP0304(){
    var root = document.getElementById('sim-ep0304-equalizacao');
    if (root) initSimEP0304(root); else setTimeout(tryInitSimEP0304, 200);
  }
  tryInitSimEP0304();
})();
</script>
</div>
""")

**Figura 3.29:** Simulador EP03_04: Ecualización de Histograma (Niveles L = 2^B)


<figure id="fig-03-sim-ep0304-equalizacao">
  <img src="imagens/fig-03-sim-ep0304-equalizacao.png" alt=" Simulador EP03_04: Ecualización de Histograma (Niveles L = 2^B) " style="max-width:80%" />
  <figcaption><strong>Figura 3.29:</strong>  Simulador EP03_04: Ecualización de Histograma (Niveles L = 2^B) </figcaption>
</figure>

In [ ]:
%%writefile EP03_04.cpp
// your solution

In [ ]:
TestSuite("EP03_04.cpp").run()

### 3.0.5 EP03_05 🔲 Aplicación de Máscara AND Binaria

En sistemas de inspección industrial por visión por computadora, es necesario aislar regiones de interés (ROI) en imágenes de piezas para verificar defectos de fabricación. La operación **AND bit a bit con una máscara binaria** es el mecanismo fundamental para recortar exactamente el área de inspección, poniendo a cero todos los píxeles fuera de ella.

Ver en [Figura 3.30](#fig-03-sim-ep0305-mascara) una simulación de este EP.


#### 3.0.5.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz de píxeles $f$ (valores $\in [0, 255]$).
3. **Máscara:** Leer la matriz binaria $m$ (valores: solo 0 o 255).
4. **Mapeo:** Para cada píxel $(i,j)$, aplicar el AND bit a bit:

$$
g(i,j) = f(i,j) \;\text{AND}\; m(i,j)
$$

donde $255 =$ `11111111` y $0 =$ `00000000` en binario.

5. **Salida:** Mostrar la matriz resultante $L \times C$.

#### 3.0.5.2 📌 Restricciones Computacionales

* **AND con 255:** $p \; \text{AND} \; 255 = p$ (todos los bits preservados).
* **AND con 0:** $p \; \text{AND} \; 0 = 0$ (todos los bits puestos a cero).
* **Máscara:** Los únicos valores posibles en la máscara son 0 y 255.
* **Implementación:** En Python, el AND bit a bit entre enteros usa el operador `&`.

#### 3.0.5.3 🧠 Fundamentación Teórica

| Píxel $f$ | Máscara $m$ | Resultado $f$ AND $m$ |
|:---------:|:-----------:|:---------------------:|
| cualquier $v$ | 255 (`11111111`) | $v$ (preservado) |
| cualquier $v$ | 0 (`00000000`) | 0 (puesto a cero) |

#### 3.0.5.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de $f$ ($L$ líneas).
* Líneas siguientes: Elementos de $m$ ($L$ líneas con valores 0 o 255).

**Salida:**

* Matriz resultante $L \times C$.

#### 3.0.5.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 2<br>3<br>100 150 200<br>50 80 120<br>255 255 0<br>0 255 255 | 100 150 0<br>0 80 120 | La máscara selecciona la región |
| 1<br>4<br>10 20 30 40<br>255 0 255 0 | 10 0 30 0 | Alternado preservado/puesto a cero |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0305-mascara" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⬛ Simulador EP03_05: Máscara AND Binaria</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f AND m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Haz clic en las celdas de la <b>Máscara m</b> para alternar entre transparente (255) y bloqueante (0), aplicando la operación lógica píxel a píxel.</p>

    <!-- Três Colunas Principais: Imagem f, Máscara m, Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Imagem f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Imagen f (0–255)</span>
        <div id="sim_ep0305_grid_f" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
      </div>

      <!-- Máscara m -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Máscara m (Clic para Alternar)</span>
        <div id="sim_ep0305_grid_mask" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↺ Reiniciar Máscara</button>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Resultado g = f AND m</span>
        <div id="sim_ep0305_grid_result" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;height:26px;display:flex;align-items:center;justify-content:center;">
          <span id="sim_ep0305_pct" style="font-size:11px;font-weight:700;color:#26241d;font-family:monospace;">—</span>
        </div>
      </div>

    </div>

    <!-- Estatísticas de Preservação -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:14px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_preserved" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>conservados
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_zeroed" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>puestos a cero
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_ratio" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>visible
      </div>
    </div>

    <!-- Legenda e Debug -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#2980b9;">255</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Transparente (conservado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#26241d;border:1.5px solid #8a8371;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#7ee7c6;">0</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Bloqueante (puesto a cero)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0305_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(i,j) = f(i,j) &amp; m(i,j)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0305(root){
    if (!root || root.dataset.simEp0305Init) return;
    root.dataset.simEp0305Init = "1";

    var gridF      = root.querySelector('#sim_ep0305_grid_f');
    var gridMask   = root.querySelector('#sim_ep0305_grid_mask');
    var gridResult = root.querySelector('#sim_ep0305_grid_result');
    var debugDiv   = root.querySelector('#sim_ep0305_debug');
    var pctSpan    = root.querySelector('#sim_ep0305_pct');
    var statPres   = root.querySelector('#sim_ep0305_stat_preserved');
    var statZero   = root.querySelector('#sim_ep0305_stat_zeroed');
    var statRatio  = root.querySelector('#sim_ep0305_stat_ratio');

    var btnNew     = root.querySelector('#sim_ep0305_btnNew');
    var btnReset   = root.querySelector('#sim_ep0305_btnReset');

    var N = 16;
    var pixels = [];
    var mask = [];

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){ return Math.floor(Math.random() * 256); });
    }

    function resetMask() {
      mask = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / 4), c = i % 4;
        return (r + c) % 2 === 0 ? 255 : 0;
      });
    }

    function render() {
      gridF.innerHTML = '';
      gridMask.innerHTML = '';
      gridResult.innerHTML = '';

      var preserved = 0, zeroed = 0;

      for (var i = 0; i < N; i++) {
        var p = pixels[i];
        var m = mask[i];
        var res = p & m;

        // Célula F
        var cellF = document.createElement('div');
        var fgF = p > 128 ? '#000000' : '#ffffff';
        cellF.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgF + ';box-sizing:border-box;';
        cellF.textContent = p;
        gridF.appendChild(cellF);

        // Célula Máscara M (interativa)
        var cellM = document.createElement('div');
        cellM.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';
        if (m === 255) {
          cellM.style.background = '#ebf4fd';
          cellM.style.border = '2px solid #2980b9';
          cellM.style.color = '#2980b9';
        } else {
          cellM.style.background = '#26241d';
          cellM.style.border = '2px solid #8a8371';
          cellM.style.color = '#7ee7c6';
        }
        cellM.textContent = m;
        cellM.title = m === 255 ? 'Clique para bloquear (0)' : 'Clique para passar (255)';

        (function(idx){
          cellM.addEventListener('click', function(){
            mask[idx] = mask[idx] === 255 ? 0 : 255;
            render();
          });
        })(i);

        gridMask.appendChild(cellM);

        // Célula Resultado G
        var cellR = document.createElement('div');
        var fgR = res > 128 ? '#000000' : '#ffffff';
        var borderStyle = m === 255 ? '2px solid #27ae60' : '1px solid #e4dcc8';
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgR + ';border:' + borderStyle + ';box-sizing:border-box;';
        cellR.textContent = res;
        gridResult.appendChild(cellR);

        if (m === 255) preserved++; else zeroed++;
      }

      var ratioPct = Math.round((preserved / N) * 100);
      statPres.textContent = preserved;
      statZero.textContent = zeroed;
      statRatio.textContent = ratioPct + '%';
      pctSpan.textContent = ratioPct + '% visível';
      debugDiv.innerHTML = 'g(i,j) = f(i,j) &amp; m(i,j) &nbsp;|&nbsp; <b>' + preserved + '</b> preservados · <b>' + zeroed + '</b> zerados';
    }

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMask();
      render();
    });

    generatePixels();
    resetMask();
    render();
  }

  function tryInitSimEP0305(){
    var root = document.getElementById('sim-ep0305-mascara');
    if (root) initSimEP0305(root); else setTimeout(tryInitSimEP0305, 200);
  }
  tryInitSimEP0305();
})();
</script>
</div>
""")

**Figura 3.30:** Simulador EP03_05: Aplicación de Máscara AND Binaria


<figure id="fig-03-sim-ep0305-mascara">
  <img src="imagens/fig-03-sim-ep0305-mascara.png" alt=" Simulador EP03_05: Aplicación de Máscara AND Binaria " style="max-width:80%" />
  <figcaption><strong>Figura 3.30:</strong>  Simulador EP03_05: Aplicación de Máscara AND Binaria </figcaption>
</figure>

In [ ]:
%%writefile EP03_05.cpp
// your solution

In [ ]:
TestSuite("EP03_05.cpp").run()

### 3.0.6 EP03_06 🌫️ Filtro de Media con *Kernel* N×N


En cámaras de vehículos autónomos, las imágenes capturadas bajo lluvia o niebla presentan ruido gaussiano. El **filtro de media** se utiliza ampliamente para su reducción en tiempo real, implementándose directamente en el **ISP** (*Image Signal Processor*) de los sensores **CMOS** (*Complementary Metal-Oxide-Semiconductor*).

Los sensores CMOS son los sensores de imagen utilizados en la mayoría de las cámaras modernas (*smartphones*, *webcams*, cámaras automotrices, etc.). Convierten la luz en señales eléctricas, y el ISP procesa estas señales en tiempo real — aplicando operaciones como reducción de ruido, balance de blancos y otros ajustes de imagen.

Ver en la [Figura 3.31](#fig-03-sim-ep0306-media) una simulación de este EP.


#### 3.0.6.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas), $C$ (columnas) y $N$ (tamaño del *kernel*, siempre impar).
2. **Datos:** Leer la matriz de píxeles $f$.
3. **Filtro de Media:** Para cada píxel $(i,j)$ **interno** (sin bordes), calcular:

$$g(i,j) = \text{round}\left(\frac{1}{N^2} \sum_{s=-(r)}^{r} \sum_{t=-(r)}^{r} f(i+s,\, j+t)\right), \quad r = \lfloor N/2 \rfloor$$

4. **Tratamiento de Bordes:** Los píxeles en el borde (donde la ventana $N \times N$ sobrepasa los límites) deben **copiarse directamente** del original sin modificación.
5. **Salida:** Mostrar la matriz resultante $L \times C$.

#### 3.0.6.2 📌 Restricciones Computacionales

* **Radio:** $r = \lfloor N/2 \rfloor$ (mitad del *kernel*, entero).
* **Píxeles internos:** $(i,j)$ con $r \le i < L-r$ y $r \le j < C-r$.
* **Redondeo:** Usar redondeo matemático antes de convertir a entero.
* **Sin *clipping*:** El promedio de valores $\in [0,255]$ permanece en $[0,255]$.

#### 3.0.6.3 🧠 Fundamentación Teórica

| Tamaño $N$ | Coeficiente | Píxeles en la ventana | Efecto |
|:-----------:|:-----------:|:---------------------:|:------:|
| 3 | $1/9 \approx 0.111$ | 9 | Suave |
| 5 | $1/25 = 0.04$ | 25 | Medio |
| 7 | $1/49 \approx 0.020$ | 49 | Fuerte |

#### 3.0.6.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $N$ (impar, $N \ge 3$).
* Líneas siguientes: Elementos de la matriz original.

**Salida:**

* Matriz filtrada $L \times C$.

#### 3.0.6.5 📌 Ejemplos
| Entrada | Salida | Observación |
|---------|--------|-------------|
| 3<br>3<br>3<br>10 20 30<br>40 50 60<br>70 80 90 | 10 20 30<br>40 50 60<br>70 80 90 | Solo borde (3×3 = borde total) |
| 5<br>5<br>3<br>0 0 0 0 0<br>0 0 0 0 0<br>0 0 100 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 0 0 0 0 | Píxel aislado: todos los 9 píxeles internos cuya ventana 3×3 incluye el valor 100 reciben round(100/9)=11 |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0306-media" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔲 Simulador EP03_06: Filtro de Media con Kernel N×N</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Media(Vecinos)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Seleccione el tamaño del kernel y pase el mouse sobre los píxeles del resultado para inspeccionar la vecindad y el cálculo de la media aritmética.</p>

    <!-- Barra de Controles / Seleção de Kernel -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Tamaño del kernel:</span>
        <button id="sim_ep0306_btn_k3" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">3 × 3 (9 vecinos)</button>
        <button id="sim_ep0306_btn_k5" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">5 × 5 (25 vecinos)</button>
      </div>
      <button id="sim_ep0306_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
    </div>

    <!-- Comparativo Lado a Lado: Original vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original (7x7) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagen Original f (7×7)</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Con ruido sal y pimienta</span>
        <div id="sim_ep0306_grid_f" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Suavizado -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Resultado g (Filtro Suavizado)</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pase el mouse para inspeccionar</span>
        <div id="sim_ep0306_grid_result" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ventana del Kernel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Inspeccionado</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0306_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno del resultado para ver el cálculo de la media.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0306(root){
    if (!root || root.dataset.simEp0306Init) return;
    root.dataset.simEp0306Init = "1";

    var ROWS = 7, COLS = 7, N = 49;
    var pixels = [], result = [], kSize = 3, radius = 1;

    var gridF   = root.querySelector('#sim_ep0306_grid_f');
    var gridRes = root.querySelector('#sim_ep0306_grid_result');
    var debug   = root.querySelector('#sim_ep0306_debug');
    var btnK3   = root.querySelector('#sim_ep0306_btn_k3');
    var btnK5   = root.querySelector('#sim_ep0306_btn_k5');
    var btnNew  = root.querySelector('#sim_ep0306_btnNew');

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){
        var v = 60 + Math.floor(Math.random() * 60);
        if (Math.random() > 0.82) v = Math.random() > 0.5 ? 255 : 0;
        return v;
      });
    }

    function calculateFilter() {
      result = pixels.slice();
      radius = Math.floor(kSize / 2);
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= radius && r < ROWS - radius && c >= radius && c < COLS - radius) {
            var sum = 0, cnt = 0;
            for (var s = -radius; s <= radius; s++) {
              for (var t = -radius; t <= radius; t++) {
                sum += pixels[(r + s) * COLS + (c + t)];
                cnt++;
              }
            }
            result[r * COLS + c] = Math.round(sum / cnt);
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightKernel(tr, tc, on) {
      var cells = gridF.children;
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var idx = r * COLS + c;
          if (!cells[idx]) continue;
          var p = pixels[idx];
          if (on && Math.abs(r - tr) <= radius && Math.abs(c - tc) <= radius) {
            cells[idx].style.background = '#fef5e7';
            cells[idx].style.color = '#b9770e';
            cells[idx].style.boxShadow = '0 0 0 2px #b9770e inset';
          } else {
            cells[idx].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[idx].style.color = textColor(p);
            cells[idx].style.boxShadow = 'none';
          }
        }
      }
    }

    function render() {
      var cols = 'repeat(' + COLS + ', 42px)';
      gridF.style.gridTemplateColumns = cols;
      gridRes.style.gridTemplateColumns = cols;
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = r < radius || r >= ROWS - radius || c < radius || c >= COLS - radius;

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '2px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel (' + row + ',' + col + ') é <b>borda</b>: valor herdado do original sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              highlightKernel(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var neighbors = [];
              for (var s = -radius; s <= radius; s++) {
                for (var t = -radius; t <= radius; t++) {
                  neighbors.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var kTotal = kSize * kSize;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): round( (' + neighbors.join(' + ') + ') / ' + kTotal + ' ) &nbsp;=&nbsp; <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightKernel(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para ver o cálculo da média.';
    }

    function setKernel(k) {
      kSize = k;
      if (k === 3) {
        btnK3.style.background = '#ebf4fd';
        btnK3.style.borderColor = '#2980b9';
        btnK3.style.color = '#2980b9';
        btnK3.style.fontWeight = '700';

        btnK5.style.background = '#f1ead7';
        btnK5.style.borderColor = '#e4dcc8';
        btnK5.style.color = '#5e5a4a';
        btnK5.style.fontWeight = '600';
      } else {
        btnK5.style.background = '#ebf4fd';
        btnK5.style.borderColor = '#2980b9';
        btnK5.style.color = '#2980b9';
        btnK5.style.fontWeight = '700';

        btnK3.style.background = '#f1ead7';
        btnK3.style.borderColor = '#e4dcc8';
        btnK3.style.color = '#5e5a4a';
        btnK3.style.fontWeight = '600';
      }
      calculateFilter();
      render();
    }

    btnK3.addEventListener('click', function(){ setKernel(3); });
    btnK5.addEventListener('click', function(){ setKernel(5); });

    btnNew.addEventListener('click', function(){
      generatePixels();
      calculateFilter();
      render();
    });

    generatePixels();
    calculateFilter();
    render();
  }

  function tryInitSimEP0306(){
    var root = document.getElementById('sim-ep0306-media');
    if (root) initSimEP0306(root); else setTimeout(tryInitSimEP0306, 200);
  }
  tryInitSimEP0306();
})();
</script>
</div>
""")

**Figura 3.31:** Simulador EP03_06: Filtro de Media con Kernel N×N


<figure id="fig-03-sim-ep0306-media">
  <img src="imagens/fig-03-sim-ep0306-media.png" alt=" Simulador EP03_06: Filtro de Media con Kernel N×N " style="max-width:80%" />
  <figcaption><strong>Figura 3.31:</strong>  Simulador EP03_06: Filtro de Media con Kernel N×N </figcaption>
</figure>

In [ ]:
%%writefile EP03_06.cpp
// your solution

In [ ]:
TestSuite("EP03_06.cpp").run()

### 3.0.7 EP03_07 🔍 Operador Laplaciano (w4) para Realce de Bordas

En tomografías de alta resolución, la nitidez de los bordes entre tejidos es crítica para el diagnóstico. El **operador Laplaciano** se utiliza ampliamente en *pipelines* de preprocesamiento de imágenes médicas para resaltar automáticamente los contornos anatómicos antes de la segmentación, evitando la intervención manual del radiólogo.

Ver en [Figura 3.32](#fig-03-sim-ep0307-laplaciano) una simulación de este EP.


#### 3.0.7.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz de píxeles $f$.
3. **Laplaciano (w4):** Para cada píxel **interno** $(i,j)$ con $1 \le i < L-1$, $1 \le j < C-1$, calcular:

$$\nabla^2 f(i,j) = f(i-1,j) + f(i+1,j) + f(i,j-1) + f(i,j+1) - 4 \cdot f(i,j)$$

4. **Realce:** Calcular la imagen realzada:

$$g(i,j) = \text{clip}(f(i,j) - \nabla^2 f(i,j))$$

5. **Borde:** Los píxeles en el borde se copian directamente: $g(i,j) = f(i,j)$.
6. **Salida:** Mostrar la matriz realzada $L \times C$.

#### 3.0.7.2 📌 Restricciones Computacionales

* ***Kernel* w4:** $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$ — solo vecinos-4.
* **Saturación:** $\text{clip}(x) = \max(0, \min(255, x))$ aplicado al resultado del realce.
* **Sin redondeo:** El Laplaciano utiliza solo sumas/restas de enteros.

#### 3.0.7.3 🧠 Fundamentación Teórica

| Región | $\nabla^2 f$ | Efecto del Realce |
|:------:|:------------:|:----------------:|
| Uniforme | $\approx 0$ | Sin alteración |
| Borde creciente | $< 0$ | Píxel aclarado |
| Borde decreciente | $> 0$ | Píxel oscurecido |

#### 3.0.7.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de la matriz original.

**Salida:**

* Matriz realzada $L \times C$.

#### 3.0.7.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|-------------|
| 3<br>3<br>0 0 0<br>0 100 0<br>0 0 0 | 0 0 0<br>0 255 0<br>0 0 0 | Pico aislado: lap=−400, g=100−(−400)=500 → clip=255 |
| 3<br>3<br>50 50 50<br>50 50 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Región uniforme: Laplaciano=0, sin alteración |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0307-laplaciano" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📐 Simulador EP03_07: Operador Laplaciano (w4)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ∓ ∇²f</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Seleccione la variante de realce y pase el mouse sobre los píxeles internos del resultado para inspeccionar la vecindad de 4 puntos y la ecuación del Laplaciano.</p>

    <!-- Barra de Controles / Seleção de Variante -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Variante:</span>
        <button id="sim_ep0307_btn_v1" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f − ∇²f (Realce Estándar)</button>
        <button id="sim_ep0307_btn_v2" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f + ∇²f (Invierte Señal)</button>
      </div>
      <button id="sim_ep0307_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuevo Escalón</button>
    </div>

    <!-- Grid Principal de Comparação (2 colunas + setas) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Imagen Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Escalón con ruido leve</span>
        <div id="sim_ep0307_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Laplaciano ∇²f -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Laplaciano ∇²f</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Bordes detectados (±128 shift)</span>
        <div id="sim_ep0307_grid_l" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Linha de Resultado g e Kernel w4 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0307_flabel">③ Resultado g = f − ∇²f</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pase el mouse para inspeccionar</span>
        <div id="sim_ep0307_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Estrutura do Kernel w4 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;font-family:monospace;">Kernel w4 (4-Vecinos)</span>
        <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#b9770e;border:1px solid #b9770e;border-radius:4px;color:#ffffff;">−4</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
        </div>
        <span style="font-size:10px;color:#8a8371;display:block;font-family:monospace;">∇²f = T + B + L + R − 4·f</span>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">4-Vecinos del Kernel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#faece7;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0307_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno del resultado para detallar la ecuación.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0307(root){
    if (!root || root.dataset.simEp0307Init) return;
    root.dataset.simEp0307Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], lapV = [], lapA = [], result = [];
    var variant = 'subtract';

    var gridF   = root.querySelector('#sim_ep0307_grid_f');
    var gridL   = root.querySelector('#sim_ep0307_grid_l');
    var gridRes = root.querySelector('#sim_ep0307_grid_res');
    var debug   = root.querySelector('#sim_ep0307_debug');
    var fLabel  = root.querySelector('#sim_ep0307_flabel');
    var btnV1   = root.querySelector('#sim_ep0307_btn_v1');
    var btnV2   = root.querySelector('#sim_ep0307_btn_v2');
    var btnNew  = root.querySelector('#sim_ep0307_btnNew');

    function generate() {
      var sc = 2 + Math.floor(Math.random() * 2);
      var dark = 40 + Math.floor(Math.random() * 30);
      var light = 160 + Math.floor(Math.random() * 40);
      pixels = Array.from({ length: N }, function(_, i) {
        var c = i % COLS;
        var v = c < sc ? dark : light;
        v += Math.floor(Math.random() * 14) - 7;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      lapV = new Array(N).fill(0);
      lapA = new Array(N).fill(0);
      result = pixels.slice();

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var t  = pixels[(r - 1) * COLS + c];
            var b  = pixels[(r + 1) * COLS + c];
            var l  = pixels[r * COLS + (c - 1)];
            var ri = pixels[r * COLS + (c + 1)];
            var f  = pixels[i];
            var lap = t + b + l + ri - 4 * f;

            lapV[i] = lap;
            lapA[i] = Math.max(0, Math.min(255, lap + 128));
            result[i] = Math.max(0, Math.min(255, variant === 'subtract' ? f - lap : f + lap));
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightCross(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i];
        cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
        cells[i].style.color = textColor(p);
        cells[i].style.boxShadow = 'none';
      }
      if (on) {
        var ci = tr * COLS + tc;
        if (cells[ci]) {
          cells[ci].style.background = '#faece7';
          cells[ci].style.color = '#c0392b';
          cells[ci].style.boxShadow = '0 0 0 2px #c0392b inset';
        }
        var neighbors = [[tr - 1, tc], [tr + 1, tc], [tr, tc - 1], [tr, tc + 1]];
        neighbors.forEach(function(n) {
          var nr = n[0], nc = n[1];
          if (nr >= 0 && nr < ROWS && nc >= 0 && nc < COLS) {
            var ni = nr * COLS + nc;
            if (cells[ni]) {
              cells[ni].style.background = '#fef5e7';
              cells[ni].style.color = '#b9770e';
              cells[ni].style.boxShadow = '0 0 0 2px #b9770e inset';
            }
          }
        });
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para detalhar a equação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridL.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = variant === 'subtract' ? '③ Resultado g = f − ∇²f' : '③ Resultado g = f + ∇²f';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], lv = lapV[i], la = lapA[i], res = result[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Laplaciano l
        var cl = document.createElement('div');
        cl.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cl.style.background = '#fafaf7';
          cl.style.color = '#8a8371';
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = '—';
        } else {
          cl.style.background = 'rgb(' + la + ',' + la + ',' + la + ')';
          cl.style.color = textColor(la);
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = lv;
        }
        gridL.appendChild(cl);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): valor herdado sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val, lapVal, origVal){
            var tVal = pixels[(row - 1) * COLS + col];
            var bVal = pixels[(row + 1) * COLS + col];
            var lVal = pixels[row * COLS + (col - 1)];
            var rVal = pixels[row * COLS + (col + 1)];

            cr.addEventListener('mouseenter', function(){
              highlightCross(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sign = variant === 'subtract' ? '−' : '+';
              debug.innerHTML = '∇²f = (' + tVal + ' + ' + bVal + ' + ' + lVal + ' + ' + rVal + ') − 4·' + origVal + ' = <b>' + lapVal + '</b> &nbsp;|&nbsp; g = clip(' + origVal + ' ' + sign + ' ' + lapVal + ') = <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightCross(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res, lv, p);
        }
        gridRes.appendChild(cr);
      }
    }

    function setVariant(v) {
      variant = v;
      if (v === 'subtract') {
        btnV1.style.background = '#fef5e7';
        btnV1.style.borderColor = '#b9770e';
        btnV1.style.color = '#b9770e';
        btnV1.style.fontWeight = '700';

        btnV2.style.background = '#f1ead7';
        btnV2.style.borderColor = '#e4dcc8';
        btnV2.style.color = '#5e5a4a';
        btnV2.style.fontWeight = '600';
      } else {
        btnV2.style.background = '#fef5e7';
        btnV2.style.borderColor = '#b9770e';
        btnV2.style.color = '#b9770e';
        btnV2.style.fontWeight = '700';

        btnV1.style.background = '#f1ead7';
        btnV1.style.borderColor = '#e4dcc8';
        btnV1.style.color = '#5e5a4a';
        btnV1.style.fontWeight = '600';
      }
      calculate();
      render();
    }

    btnV1.addEventListener('click', function(){ setVariant('subtract'); });
    btnV2.addEventListener('click', function(){ setVariant('add'); });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0307(){
    var root = document.getElementById('sim-ep0307-laplaciano');
    if (root) initSimEP0307(root); else setTimeout(tryInitSimEP0307, 200);
  }
  tryInitSimEP0307();
})();
</script>
</div>
""")

**Figura 3.32:** Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas


<figure id="fig-03-sim-ep0307-laplaciano">
  <img src="imagens/fig-03-sim-ep0307-laplaciano.png" alt=" Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas " style="max-width:80%" />
  <figcaption><strong>Figura 3.32:</strong>  Simulador EP03_07: Operador Laplaciano (w4) para Realce de Bordas </figcaption>
</figure>

In [ ]:
%%writefile EP03_07.cpp
// your solution

In [ ]:
TestSuite("EP03_07.cpp").run()

### 3.0.8 EP03_08 🧭 Gradiente de Sobel: Gx y Gy

En robots exploradores de Marte (como el Perseverance), la detección de obstáculos se realiza en tiempo real mediante cámaras estereoscópicas. El **operador de Sobel** calcula el gradiente direccional de la escena y se utiliza en el algoritmo de detección de bordes para identificar rocas, fisuras y desniveles del terreno que puedan comprometer la navegación.

Ver en [Figura 3.33](#fig-03-sim-ep0308-sobel) una simulación de este EP.


#### 3.0.8.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz $f$.
3. **Gx y Gy:** Para cada píxel **interno** $(i,j)$ con $1 \le i < L-1$, $1 \le j < C-1$:

$$G_x(i,j) = [f(i-1,j+1) + 2f(i,j+1) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i,j-1) + f(i+1,j-1)]$$

$$G_y(i,j) = [f(i+1,j-1) + 2f(i+1,j) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i-1,j) + f(i-1,j+1)]$$

4. **Magnitud:** $|\nabla f(i,j)| = \text{clip}(\text{round}(\sqrt{G_x^2 + G_y^2}))$.
5. **Borde:** Los píxeles de borde reciben magnitud 0.
6. **Salida:** Mostrar la magnitud $L \times C$.

#### 3.0.8.2 📌 Restricciones Computacionales

* **Redondeo:** Aplicar `round` antes de convertir a entero.
* **Saturación:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Raíz cuadrada:** Usar $\sqrt{G_x^2 + G_y^2}$ (no la aproximación $|G_x| + |G_y|$).

#### 3.0.8.3 🧠 Fundamentación Teórica

| Operador | Detecta | Coeficientes diagonales |
|:--------:|:-------:|:----------------------:|
| $G_x$ | Bordes verticales | $\pm 1$ |
| $G_y$ | Bordes horizontales | $\pm 1$ |
| $|\nabla f|$ | Todos los bordes | Combinado |

#### 3.0.8.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de la matriz.

**Salida:**

* Magnitud del gradiente, matriz $L \times C$.

#### 3.0.8.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 0 0<br>0 0 0 | 0 0 0<br>0 0 0<br>0 0 0 | Imagen nula: gradiente cero |
| 3<br>3<br>0 0 255<br>0 0 255<br>0 0 255 | 0 0 0<br>0 255 0<br>0 0 0 | Borde vertical central: Gx alto |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0308-sobel" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧭 Simulador EP03_08: Gradiente de Sobel (Gx y Gy)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">|∇f| = √(Gx² + Gy²)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Analice la descomposición horizontal (Gx) y vertical (Gy) del operador de Sobel y pase el mouse sobre los píxeles de la magnitud para inspeccionar la vecindad 3×3.</p>

    <!-- Barra de Controles / Kernels Explicativos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
        <button id="sim_ep0308_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Escena</button>
        <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Kernels de Sobel:</span>
      </div>

      <!-- Representação Visual dos Kernels Gx e Gy -->
      <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
        
        <!-- Kernel Gx -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#2980b9;font-family:monospace;">Gx</span>
        </div>

        <!-- Kernel Gy -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#854f0b;color:#fac775;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ef9f27;color:#412402;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#b9770e;font-family:monospace;">Gy</span>
        </div>

      </div>

    </div>

    <!-- Comparativo em 2 Linhas (Original, Magnitude, Gx, Gy) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagen Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Matriz 5×5 píxeles</span>
        <div id="sim_ep0308_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Magnitude do Gradiente |∇f| -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Magnitud |∇f|</span>
        <span style="font-size:10px;color:#27ae60;display:block;margin-bottom:10px;">√(Gx² + Gy²)</span>
        <div id="sim_ep0308_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Gradiente Horizontal Gx -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gx — Gradiente Horizontal</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Azul = Negativo · Blanco = Cero · Azul Vivo = Positivo</span>
        <div id="sim_ep0308_grid_gx" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente Vertical Gy -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gy — Gradiente Vertical</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Ámbar = Negativo · Blanco = Cero · Ámbar Vivo = Positivo</span>
        <div id="sim_ep0308_grid_gy" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Vecindad 3×3 Inspeccionada</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #8a8371;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Forzado a 0)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0308_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno de la magnitud para ver la descomposición Gx y Gy.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0308(root){
    if (!root || root.dataset.simEp0308Init) return;
    root.dataset.simEp0308Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], gxV = [], gyV = [], mgV = [];

    var gridF  = root.querySelector('#sim_ep0308_grid_f');
    var gridGx = root.querySelector('#sim_ep0308_grid_gx');
    var gridGy = root.querySelector('#sim_ep0308_grid_gy');
    var gridM  = root.querySelector('#sim_ep0308_grid_m');
    var debug  = root.querySelector('#sim_ep0308_debug');
    var btnNew = root.querySelector('#sim_ep0308_btnNew');

    function generate() {
      var block = Math.random() > 0.3;
      var bg = 30 + Math.floor(Math.random() * 30);
      var obj = 180 + Math.floor(Math.random() * 50);

      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = bg;
        if (block && r >= 2 && r <= 3 && c >= 2 && c <= 3) v = obj;
        else if (!block && r + c >= 4) v = obj - 40;
        v += Math.floor(Math.random() * 10) - 5;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      gxV = new Array(N).fill(0);
      gyV = new Array(N).fill(0);
      mgV = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var p = [
              pixels[(r - 1) * COLS + (c - 1)], pixels[(r - 1) * COLS + c], pixels[(r - 1) * COLS + (c + 1)],
              pixels[r * COLS + (c - 1)],       pixels[r * COLS + c],       pixels[r * COLS + (c + 1)],
              pixels[(r + 1) * COLS + (c - 1)], pixels[(r + 1) * COLS + c], pixels[(r + 1) * COLS + (c + 1)]
            ];
            var gx = (p[2] + 2 * p[5] + p[8]) - (p[0] + 2 * p[3] + p[6]);
            var gy = (p[6] + 2 * p[7] + p[8]) - (p[0] + 2 * p[1] + p[2]);

            gxV[i] = gx;
            gyV[i] = gy;
            mgV[i] = Math.min(255, Math.round(Math.sqrt(gx * gx + gy * gy)));
          }
        }
      }
    }

    function textColor(g){ return g > 150 ? '#000000' : '#ffffff'; }

    function colorGxBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        var r = Math.round(4 + t * 20);
        var g = Math.round(44 + t * 71);
        var b = Math.round(83 + t * 89);
        return 'rgb(' + r + ',' + g + ',' + b + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(12 + t * 0) + ',' + Math.round(68 - t * 24) + ',' + Math.round(165 - t * 82) + ')';
      }
    }

    function colorGyBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        return 'rgb(' + Math.round(186 + t * 63) + ',' + Math.round(117 + t * 70) + ',' + Math.round(23 - t * 18) + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(99 + t * 0) + ',' + Math.round(56 - t * 20) + ',' + Math.round(11 - t * 5) + ')';
      }
    }

    function textSignedColor(bg) {
      if (bg === '#fafaf7') return '#26241d';
      var m = bg.match(/rgb\((\d+),(\d+),(\d+)\)/);
      if (!m) return '#ffffff';
      var lum = 0.299 * m[1] + 0.587 * m[2] + 0.114 * m[3];
      return lum > 140 ? '#000000' : '#ffffff';
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.';
    }

    function render() {
      gridF.innerHTML = '';
      gridGx.innerHTML = '';
      gridGy.innerHTML = '';
      gridM.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], gx = gxV[i], gy = gyV[i], mag = mgV[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Gx
        var cgx = document.createElement('div');
        cgx.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgx.style.background = '#fafaf7';
          cgx.style.color = '#8a8371';
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = '—';
        } else {
          var bgGx = colorGxBetter(gx);
          cgx.style.background = bgGx;
          cgx.style.color = textSignedColor(bgGx);
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = (gx > 0 ? '+' : '') + gx;
        }
        gridGx.appendChild(cgx);

        // Célula Gy
        var cgy = document.createElement('div');
        cgy.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgy.style.background = '#fafaf7';
          cgy.style.color = '#8a8371';
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = '—';
        } else {
          var bgGy = colorGyBetter(gy);
          cgy.style.background = bgGy;
          cgy.style.color = textSignedColor(bgGy);
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = (gy > 0 ? '+' : '') + gy;
        }
        gridGy.appendChild(cgy);

        // Célula Magnitude m
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';
        if (isBorder) {
          cm.style.background = '#26241d';
          cm.style.border = '1.5px solid #8a8371';
          cm.style.color = '#7ee7c6';
          cm.textContent = '0';

          (function(row, col){
            cm.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): sem vizinhança completa → forçado para <b>0</b>';
            });
            cm.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c);
        } else {
          cm.style.background = 'rgb(' + mag + ',' + mag + ',' + mag + ')';
          cm.style.color = textColor(mag);
          cm.style.border = '1px solid #e4dcc8';
          cm.style.cursor = 'pointer';
          cm.textContent = mag;

          (function(row, col, gxv, gyv, mgv){
            cm.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cm.style.transform = 'scale(1.12)';
              cm.style.boxShadow = '0 0 0 2px #27ae60 inset';
              cm.style.background = '#eafaf1';
              cm.style.color = '#27ae60';

              debug.innerHTML = 'Pixel (' + row + ',' + col + '): Gx = <b>' + gxv + '</b> &nbsp;|&nbsp; Gy = <b>' + gyv + '</b> &nbsp;|&nbsp; |∇f| = round(√(' + gxv + '² + ' + gyv + '²)) = <b>' + mgv + '</b>';
            });

            cm.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cm.style.transform = 'scale(1)';
              cm.style.boxShadow = 'none';
              cm.style.background = 'rgb(' + mgv + ',' + mgv + ',' + mgv + ')';
              cm.style.color = textColor(mgv);
              resetDebug();
            });
          })(r, c, gx, gy, mag);
        }
        gridM.appendChild(cm);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0308(){
    var root = document.getElementById('sim-ep0308-sobel');
    if (root) initSimEP0308(root); else setTimeout(tryInitSimEP0308, 200);
  }
  tryInitSimEP0308();
})();
</script>
</div>
""")

**Figura 3.33:** Simulador EP03_08: Gradiente de Sobel (Gx y Gy)


<figure id="fig-03-sim-ep0308-sobel">
  <img src="imagens/fig-03-sim-ep0308-sobel.png" alt=" Simulador EP03_08: Gradiente de Sobel (Gx y Gy) " style="max-width:80%" />
  <figcaption><strong>Figura 3.33:</strong>  Simulador EP03_08: Gradiente de Sobel (Gx y Gy) </figcaption>
</figure>

In [ ]:
%%writefile EP03_08.cpp
// your solution

In [ ]:
TestSuite("EP03_08.cpp").run()

### 3.0.9 EP03_09 📡 Filtro de la Mediana 3×3

Las imágenes de radar de apertura sintética (SAR) utilizadas en monitoreo ambiental y militar sufren de un tipo específico de ruido llamado *speckle*, que posee características similares al ruido sal y pimienta. El **filtro de la mediana** es el método estándar para eliminar este ruido porque preserva los bordes de las estructuras mientras elimina los puntos espurios.

Ver en [Figura 3.34](#fig-03-sim-ep0309-mediana) una simulación de este EP.


#### 3.0.9.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Datos:** Leer la matriz de píxeles $f$.
3. **Filtro de la Mediana 3×3:** Para cada píxel **interno** $(i,j)$ con $1 \le i < L-1$, $1 \le j < C-1$:
   - Recolectar los 9 píxeles de la vecindad $3 \times 3$: $\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$.
   - Ordenar los 9 valores en orden creciente.
   - Asignar $g(i,j)$ al valor central (posición índice 4, considerando índice 0).

$$g(i,j) = \text{mediana}\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$$

4. **Borde:** Copiar directamente: $g(i,j) = f(i,j)$.
5. **Salida:** Mostrar la matriz filtrada $L \times C$.

#### 3.0.9.2 📌 Restricciones Computacionales

* **Ventana:** Siempre $3 \times 3 = 9$ elementos.
* **Mediana:** El elemento central de la secuencia ordenada (índice 4 de 0 a 8).
* **Sin recorte:** La mediana de valores en $[0, 255]$ permanece en $[0, 255]$.
* **No lineal:** El filtro de mediana no puede expresarse como convolución lineal.

#### 3.0.9.3 🧠 Fundamentación Teórica

| Ruido | Filtro de Media | Filtro de Mediana |
|:-----:|:---------------:|:-----------------:|
| Sal y pimienta (0 o 255) | Dispersa el ruido | Elimina sin distorsionar bordes |
| Gaussiano | Reduce eficazmente | Reduce parcialmente |

#### 3.0.9.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos de la matriz.

**Salida:**

* Matriz filtrada $L \times C$.

#### 3.0.9.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 3<br>3<br>100 100 100<br>100 0 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | Punto negro eliminado: mediana de 8×100+1×0 = 100 |
| 3<br>3<br>50 50 50<br>50 255 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Punto blanco (sal) eliminado |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0309-mediana" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📉 Simulador EP03_09: Filtro de la Mediana 3×3</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Mediana(Vecinos)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Inyecte ruido impulsivo (sal y pimienta) y pase el mouse sobre los píxeles internos del resultado para inspeccionar la ordenación del vector de vecindad y la eliminación del ruido.</p>

    <!-- Barra de Controles / Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <button id="sim_ep0309_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Inyectar Ruido Impulsivo</button>
      <span style="font-size:11px;color:#8a8371;">Ruido de sal (255) y pimienta (0) — ~30% de los píxeles internos afectados</span>
    </div>

    <!-- Comparativo Lado a Lado: Original com Ruído vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f com Ruído -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Imagen f — Con Ruido</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Sal (255) y pimienta (0) visibles</span>
        <div id="sim_ep0309_grid_f" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g sem Ruído -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Resultado g — Sin Ruido</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pase el mouse para inspeccionar</span>
        <div id="sim_ep0309_grid_result" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Vetor Ordenado -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:14px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:3px;">Vector de Vecindad 3×3 — Ordenado</span>
      <span style="font-size:10.5px;color:#8a8371;display:block;margin-bottom:8px;">Pase el mouse sobre un píxel interno del resultado para visualizar</span>
      <div id="sim_ep0309_vector" style="display:flex;flex-wrap:wrap;justify-content:center;gap:3px;min-height:32px;padding:4px 0;align-items:center;">
        <span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>
      </div>
    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ventana 3×3</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Mediana</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Ruido (Eliminado)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0309_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pase el mouse sobre un píxel interno del resultado para ver el proceso de ordenación.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0309(root){
    if (!root || root.dataset.simEp0309Init) return;
    root.dataset.simEp0309Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], result = [];

    var gridF  = root.querySelector('#sim_ep0309_grid_f');
    var gridRes= root.querySelector('#sim_ep0309_grid_result');
    var vector = root.querySelector('#sim_ep0309_vector');
    var debug  = root.querySelector('#sim_ep0309_debug');
    var btnNew = root.querySelector('#sim_ep0309_btnNew');

    function generate() {
      pixels = Array.from({ length: N }, function(){
        var v = 110 + Math.floor(Math.random() * 30);
        var rnd = Math.random();
        if (rnd > 0.82) v = 255;
        else if (rnd < 0.18) v = 0;
        return v;
      });
      calculate();
    }

    function calculate() {
      result = pixels.slice();
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var win = [];
            for (var s = -RADIUS; s <= RADIUS; s++) {
              for (var t = -RADIUS; t <= RADIUS; t++) {
                win.push(pixels[(r + s) * COLS + (c + t)]);
              }
            }
            win.sort(function(a, b){ return a - b; });
            result[r * COLS + c] = win[4];
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }
    function isNoise(v){ return v === 0 || v === 255; }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          if (isNoise(p)) {
            cells[i].style.background = p === 255 ? '#ffffff' : '#26241d';
            cells[i].style.color = p === 255 ? '#c0392b' : '#e74c3c';
            cells[i].style.border = '2px solid #c0392b';
          } else {
            cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[i].style.color = textColor(p);
            cells[i].style.border = '1px solid #e4dcc8';
          }
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function showVector(raw, sorted, median) {
      vector.innerHTML = '';

      var rawLabel = document.createElement('span');
      rawLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      rawLabel.textContent = 'Bruto:';
      vector.appendChild(rawLabel);

      raw.forEach(function(v){
        var d = document.createElement('div');
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        d.style.background = noise ? '#26241d' : '#fafaf7';
        d.style.color = noise ? '#e74c3c' : '#26241d';
        d.style.border = noise ? '1.5px solid #c0392b' : '1px solid #e4dcc8';
        d.textContent = v;
        vector.appendChild(d);
      });

      var arr = document.createElement('span');
      arr.style.cssText = 'font-size:14px;margin:0 6px;color:#8a8371;font-weight:700;';
      arr.textContent = '→';
      vector.appendChild(arr);

      var sortLabel = document.createElement('span');
      sortLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      sortLabel.textContent = 'Ordenado:';
      vector.appendChild(sortLabel);

      sorted.forEach(function(v, i){
        var d = document.createElement('div');
        var isMedian = (i === 4);
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        if (isMedian) {
          d.style.background = '#eafaf1';
          d.style.color = '#27ae60';
          d.style.border = '2px solid #27ae60';
        } else if (noise) {
          d.style.background = '#26241d';
          d.style.color = '#e74c3c';
          d.style.border = '1.5px solid #c0392b';
        } else {
          d.style.background = '#fafaf7';
          d.style.color = '#26241d';
          d.style.border = '1px solid #e4dcc8';
        }
        d.textContent = v;
        vector.appendChild(d);
      });

      var eq = document.createElement('span');
      eq.style.cssText = 'font-size:11px;margin-left:8px;font-family:monospace;color:#27ae60;font-weight:700;';
      eq.innerHTML = '→ mediana = <b>' + median + '</b>';
      vector.appendChild(eq);
    }

    function resetVector() {
      vector.innerHTML = '<span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>';
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para ver o processo de ordenação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);
        var noise = isNoise(p);

        // Célula F com Ruído
        var cf = document.createElement('div');
        if (noise) {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:' + (p === 255 ? '#ffffff' : '#26241d') + ';color:' + (p === 255 ? '#c0392b' : '#e74c3c') + ';border:2px solid #c0392b;box-sizing:border-box;';
        } else {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        }
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem filtro → <b>' + val + '</b>';
              resetVector();
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var raw = [];
              for (var s = -RADIUS; s <= RADIUS; s++) {
                for (var t = -RADIUS; t <= RADIUS; t++) {
                  raw.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var sorted = raw.slice().sort(function(a, b){ return a - b; });
              showVector(raw, sorted, resVal);

              var noiseCount = raw.filter(isNoise).length;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): ' + noiseCount + ' vizinho(s) com ruído na janela &nbsp;→&nbsp; mediana = posição [4] = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
              resetVector();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0309(){
    var root = document.getElementById('sim-ep0309-mediana');
    if (root) initSimEP0309(root); else setTimeout(tryInitSimEP0309, 200);
  }
  tryInitSimEP0309();
})();
</script>
</div>
""")

**Figura 3.34:** Simulador EP03_09: Filtro de la Mediana 3×3


<figure id="fig-03-sim-ep0309-mediana">
  <img src="imagens/fig-03-sim-ep0309-mediana.png" alt=" Simulador EP03_09: Filtro de la Mediana 3×3 " style="max-width:80%" />
  <figcaption><strong>Figura 3.34:</strong>  Simulador EP03_09: Filtro de la Mediana 3×3 </figcaption>
</figure>

In [ ]:
%%writefile EP03_09.cpp
// your solution

In [ ]:
TestSuite("EP03_09.cpp").run()

### 3.0.10 EP03_10 ✨ *Unsharp Masking* (USM)

En los sistemas de digitalización de documentos históricos y obras de arte, la nitidez de las imágenes es fundamental para la lectura de textos manuscritos y detalles ornamentales. El ***Unsharp Masking* (USM)** es el algoritmo de realce de nitidez estándar utilizado en *escáneres* profesionales y software como Adobe Photoshop, controlado por el parámetro $k$ que determina la intensidad del realce.

Ver en la [Figura 3.35](#fig-03-sim-ep0310-unsharp) una simulación de este EP.


#### 3.0.10.1 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (líneas) y $C$ (columnas).
2. **Parámetro:** Leer el valor real $k$ (intensidad del realce, $k \ge 0$).
3. **Datos:** Leer la matriz de píxeles $f$.
4. **Suavizado:** Calcular $\bar{f}$ con el filtro de promedio $3\times3$ (solo píxeles internos; bordes mantenidos):

$$\bar{f}(i,j) = \frac{1}{9} \sum_{s=-1}^{1} \sum_{t=-1}^{1} f(i+s, j+t)$$

5. **Máscara de alta frecuencia:** $m(i,j) = f(i,j) - \bar{f}(i,j)$.
6. **Realce USM:** Para cada píxel interno:

$$g(i,j) = \text{clip}\left(\text{round}\left(f(i,j) + k \cdot m(i,j)\right)\right)$$

7. **Borde:** $g(i,j) = f(i,j)$ (copia directa).
8. **Salida:** Mostrar la matriz realzada $L \times C$.

#### 3.0.10.2 📌 Restricciones Computacionales

* **Redondeo:** Aplicar `round` antes del clip.
* **Saturación:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operaciones en float:** Calcular $\bar{f}$ y $m$ en punto flotante antes de redondear el resultado final.
* **$k = 0$:** Sin realce — la salida es idéntica a la entrada (excepto en los bordes).

#### 3.0.10.3 🧠 Fundamentación Teórica

| Etapa | Operación | Descripción |
|:-----:|:----------|:------------|
| 1 | $\bar{f} = f * \frac{1}{9}\mathbf{1}_{3\times3}$ | Suavizado (bajas frecuencias) |
| 2 | $m = f - \bar{f}$ | Máscara (altas frecuencias) |
| 3 | $g = \text{clip}(\text{round}(f + k \cdot m))$ | Realce ponderado |

#### 3.0.10.4 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Real $k$.
* Líneas siguientes: Elementos de la matriz original.

**Salida:**

* Matriz realzada $L \times C$.

#### 3.0.10.5 📌 Ejemplos

| Entrada | Salida | Observación |
|---------|--------|-------------|
| 3<br>3<br>0.0<br>100 100 100<br>100 100 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | k=0: sin realce |
| 3<br>3<br>1.0<br>50 50 50<br>50 200 50<br>50 50 50 | 50 50 50<br>50 255 50<br>50 50 50 | k=1: píxel central realzado y saturado |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0310-unsharp" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">✨ Simulador EP03_10: Enmascaramiento Unsharp (USM)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k · m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajusta el factor de ganancia k, observa el pipeline completo de realce (desenfoque, máscara de alta frecuencia) y pasa el mouse sobre el resultado.</p>

    <!-- Barra de Controles / Slider de Ganho k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:10px;flex:1;min-width:240px;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Factor de ganancia k:</span>
        <input id="sim_ep0310_k" type="range" min="0.0" max="3.0" step="0.5" value="1.0" style="flex:1;cursor:pointer;accent-color:#2980b9;">
        <span id="sim_ep0310_klabel" style="font-family:monospace;font-size:11px;font-weight:700;background:#26241d;color:#7ee7c6;padding:3px 8px;border-radius:6px;">k = 1.0</span>
      </div>

      <button id="sim_ep0310_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nueva Imagen</button>
    </div>

    <!-- Comparativo do Pipeline USM (Grid de 4 Cards) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ① Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Imagen Original f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Matriz 5×5 píxeles</span>
        <div id="sim_ep0310_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ② Desfocado f_barra -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Desenfocado f̄</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Promedio 3 × 3</span>
        <div id="sim_ep0310_grid_b" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ③ Máscara m -->
      <div style="background:#fafaf7;border:2px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">③ Máscara m</span>
        <span style="font-size:10px;color:#c0392b;display:block;margin-bottom:10px;">m = f − f̄ (Altas Frecuencias)</span>
        <div id="sim_ep0310_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ④ Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0310_flabel">④ Resultado g = f + 1.0·m</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Pasa el mouse para inspeccionar</span>
        <div id="sim_ep0310_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Leyenda:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Vecindario 3×3</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Píxel Central</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Borde (Copiado)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fbeaf0;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Máscara Positiva/Negativa</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0310_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Pasa el mouse sobre un píxel interno del resultado para rastrear el pipeline completo.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0310(root){
    if (!root || root.dataset.simEp0310Init) return;
    root.dataset.simEp0310Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], blurred = [], mask = [], result = [];
    var gainK = 1.0;

    var gridF  = root.querySelector('#sim_ep0310_grid_f');
    var gridB  = root.querySelector('#sim_ep0310_grid_b');
    var gridM  = root.querySelector('#sim_ep0310_grid_m');
    var gridRes= root.querySelector('#sim_ep0310_grid_res');
    var debug  = root.querySelector('#sim_ep0310_debug');
    var fLabel = root.querySelector('#sim_ep0310_flabel');
    var sliderK= root.querySelector('#sim_ep0310_k');
    var kLabel = root.querySelector('#sim_ep0310_klabel');
    var btnNew = root.querySelector('#sim_ep0310_btnNew');

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function colorMask(v) {
      if (Math.abs(v) < 2) return '#fafaf7';
      if (v > 0) {
        var t = Math.min(1, v / 80);
        return 'rgb(' + Math.round(230 + t * 20) + ',' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 60) + ')';
      }
      var t = Math.min(1, -v / 80);
      return 'rgb(' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 40) + ',' + Math.round(230 + t * 20) + ')';
    }

    function generate() {
      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = (r >= 2 && c >= 2) ? 170 : 70;
        v += Math.floor(Math.random() * 16) - 8;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      blurred = new Array(N).fill(0);
      mask = new Array(N).fill(0);
      result = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var sum = 0;
            for (var dr = -RADIUS; dr <= RADIUS; dr++) {
              for (var dc = -RADIUS; dc <= RADIUS; dc++) {
                sum += pixels[(r + dr) * COLS + (c + dc)];
              }
            }
            var b = sum / 9;
            var m = pixels[i] - b;
            blurred[i] = b;
            mask[i] = m;
            result[i] = Math.max(0, Math.min(255, Math.round(pixels[i] + gainK * m)));
          } else {
            blurred[i] = pixels[i];
            mask[i] = 0;
            result[i] = pixels[i];
          }
        }
      }
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para rastrear o pipeline completo.';
    }

    function render() {
      gridF.innerHTML = '';
      gridB.innerHTML = '';
      gridM.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = '④ Resultado g = f +' + gainK.toFixed(1) + '·m';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], b = blurred[i], m = mask[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);

        // Célula F
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Desfocado B
        var cb = document.createElement('div');
        cb.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cb.style.background = '#fafaf7';
          cb.style.color = '#8a8371';
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = '—';
        } else {
          var bv = Math.round(b);
          cb.style.background = 'rgb(' + bv + ',' + bv + ',' + bv + ')';
          cb.style.color = textColor(bv);
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = b.toFixed(1);
        }
        gridB.appendChild(cb);

        // Célula Máscara M
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cm.style.background = '#fafaf7';
          cm.style.color = '#8a8371';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = '0';
        } else {
          var sg = m >= 0 ? '+' : '';
          cm.style.background = colorMask(m);
          cm.style.color = '#26241d';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = sg + m.toFixed(1);
        }
        gridM.appendChild(cm);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem alteração &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, pVal, bVal, mVal, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sg = mVal >= 0 ? '+' : '';
              var raw = pVal + gainK * mVal;
              debug.innerHTML = '① f=' + pVal + ' &nbsp;→&nbsp; ② f̄=' + bVal.toFixed(1) + ' &nbsp;→&nbsp; ③ m=f−f̄=' + sg + mVal.toFixed(1) + ' &nbsp;→&nbsp; ④ g = clip(' + pVal + ' + ' + gainK.toFixed(1) + '·(' + sg + mVal.toFixed(1) + ')) = clip(' + raw.toFixed(1) + ') = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
            });
          })(r, c, p, b, m, res);
        }
        gridRes.appendChild(cr);
      }
    }

    sliderK.addEventListener('input', function(e){
      gainK = parseFloat(e.target.value) || 0;
      kLabel.textContent = 'k = ' + gainK.toFixed(1);
      calculate();
      render();
    });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0310(){
    var root = document.getElementById('sim-ep0310-unsharp');
    if (root) initSimEP0310(root); else setTimeout(tryInitSimEP0310, 200);
  }
  tryInitSimEP0310();
})();
</script>
</div>
""")

**Figura 3.35:** Simulador EP03_10: *Máscara de enfoque* (USM)


<figure id="fig-03-sim-ep0310-unsharp">
  <img src="imagens/fig-03-sim-ep0310-unsharp.png" alt=" Simulador EP03_10: *Máscara de enfoque* (USM) " style="max-width:80%" />
  <figcaption><strong>Figura 3.35:</strong>  Simulador EP03_10: *Máscara de enfoque* (USM) </figcaption>
</figure>

In [ ]:
%%writefile EP03_10.cpp
// your solution

In [ ]:
TestSuite("EP03_10.cpp").run()

## Referências do Capítulo


BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.